**DP-GBC: Differentially Private Tabular Data Synthesis via Granular-Ball Computing**

In [ ]:
# Cell 1: Imports and environment

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                              balanced_accuracy_score, average_precision_score)
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

print("Accountant + environment ready. PrivacyLedger is defined in Cell 4c.")

In [ ]:
# Cell 2 (Kaggle version): full dataset loader, no Google Drive
import socket
import time
import os

from sklearn.datasets import load_breast_cancer, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
import pandas as pd
import numpy as np

# Set a longer default socket timeout to tolerate slow responses
socket.setdefaulttimeout(180)

# Use Kaggle's local writable storage for OpenML cache
CACHE_HOME = "/kaggle/working/openml_cache"
os.makedirs(CACHE_HOME, exist_ok=True)

# Optional: define seed if you don't already have one
SEED = 42

def fetch_openml_safe(data_id, retries=6, base_delay=15):
    """Fetch OpenML dataset with retries, exponential backoff, and local caching."""
    last_err = None
    for attempt in range(retries):
        try:
            try:
                return fetch_openml(
                    data_id=data_id,
                    as_frame=True,
                    parser="auto",
                    data_home=CACHE_HOME
                )
            except TypeError:
                # Older sklearn versions may not support parser="auto"
                return fetch_openml(
                    data_id=data_id,
                    as_frame=True,
                    data_home=CACHE_HOME
                )
        except Exception as e:
            last_err = e
            wait = base_delay * (2 ** attempt)
            print(
                f"  attempt {attempt+1}/{retries} failed "
                f"({type(e).__name__}: {e}); retrying in {wait}s"
            )
            time.sleep(wait)
    raise last_err

def check_openml_availability():
    """
    Quick diagnostic: try to fetch the tiny iris dataset (ID 61).
    If this fails, OpenML is globally unreachable or heavily throttled.
    """
    print("Running pre-flight OpenML check with iris (ID 61)...")
    try:
        old_timeout = socket.getdefaulttimeout()
        socket.setdefaulttimeout(20)
        _ = fetch_openml(
            data_id=61,
            as_frame=True,
            parser="auto",
            data_home=CACHE_HOME
        )
        socket.setdefaulttimeout(old_timeout)
        print("✅ OpenML is reachable (iris loaded). Proceeding with normal retries.")
        return True
    except TypeError:
        try:
            socket.setdefaulttimeout(20)
            _ = fetch_openml(
                data_id=61,
                as_frame=True,
                data_home=CACHE_HOME
            )
            socket.setdefaulttimeout(old_timeout)
            print("✅ OpenML is reachable (iris loaded). Proceeding with normal retries.")
            return True
        except Exception as e:
            socket.setdefaulttimeout(old_timeout)
            print(f"❌ OpenML check failed: {type(e).__name__}: {e}")
            print("OpenML appears to be down or heavily throttled. Reducing retries to fail fast.")
            return False
    except Exception as e:
        socket.setdefaulttimeout(old_timeout)
        print(f"❌ OpenML check failed: {type(e).__name__}: {e}")
        print("OpenML appears to be down or heavily throttled. Reducing retries to fail fast.")
        return False

# Determine retry count based on availability
openml_available = check_openml_availability()
if openml_available:
    OPENML_RETRIES = 4
    OPENML_BASE_DELAY = 10
else:
    OPENML_RETRIES = 2
    OPENML_BASE_DELAY = 5

def prepare_tabular(X, y, name="", max_rows=None, seed=SEED):
    X = X.copy()
    y = pd.Series(y).astype(str).str.strip()
    na_vals = ["?", "NA", "N/A", "na", "n/a", "null", "None", "", "nan"]
    y = y.replace(na_vals, np.nan)
    mask = y.notna()
    X, y = X[mask].reset_index(drop=True), y[mask].reset_index(drop=True)

    for col in X.columns:
        converted = pd.to_numeric(X[col], errors="coerce")
        if converted.notna().mean() > 0.90:
            X[col] = converted
        else:
            cat = X[col].astype(str).str.strip().replace(na_vals, np.nan)
            X[col] = pd.Categorical(cat).codes.astype(float)
            X.loc[cat.isna(), col] = np.nan

    X = X.dropna()
    y = y.loc[X.index].reset_index(drop=True)
    X = X.reset_index(drop=True)
    X = X.loc[:, X.nunique(dropna=True) > 1]  # drop constant columns

    y_enc = pd.factorize(y)[0]
    y_enc = pd.Series(y_enc, name="target")

    if len(X) < 100 or pd.Series(y_enc).nunique() < 2:
        raise ValueError(f"{name}: too few samples or single class after cleaning")

    if max_rows is not None and len(X) > max_rows:
        X, _, y_enc, _ = train_test_split(
            X,
            y_enc,
            train_size=max_rows / len(X),
            random_state=seed,
            stratify=y_enc
        )
        X, y_enc = X.reset_index(drop=True), y_enc.reset_index(drop=True)

    return X, y_enc

datasets = {}
CATEGORY_CARDINALITY = {}   # <-- NEW: store ARFF-declared category counts

def add_dataset(name, X, y, max_rows=None, category_cards=None):
    try:
        Xp, yp = prepare_tabular(X, y, name=name, max_rows=max_rows)
        datasets[name] = (Xp, yp)
        if category_cards:
            CATEGORY_CARDINALITY[name] = category_cards
        bal = yp.value_counts(normalize=True).round(3).to_dict()
        print(
            f"OK  {name:<15} n={len(Xp):<6} d={Xp.shape[1]:<3} "
            f"classes={yp.nunique()} balance={bal}"
        )
    except Exception as e:
        print(f"SKIP {name:<15} reason: {e}")

bc = load_breast_cancer()
add_dataset("BreastCancer", pd.DataFrame(bc.data, columns=bc.feature_names), bc.target)

# real OpenML IDs, same ones from your original working run
openml_sources = [
    (37,   "PimaDiabetes",     None),
    (49,   "HeartHungarian",   None),
    (53,   "HeartStatlog",     None),
    (31,   "CreditG",          None),
    (1590, "AdultIncome",      6000),
    (1464, "BloodTransfusion", None),
    (1120, "MAGIC04",          6000),
    (1462, "Banknote",         None),
    (4538, "CervicalCancer",   None),
    (1597, "CreditFraud",      8000),
    (1461, "BankMarketing",    6000),
    (150,  "Covertype",        6000),
    # NEW fully numeric datasets to restore statistical power
    (59,   "Ionosphere",       None),
    (40,   "Sonar",            None),
    (187,  "Wine",             None),
    (15,   "WisconsinOriginal",None),   # original Wisconsin breast cancer (9 features)
    (60,   "Waveform",         None),
    (54,   "Vehicle",          None),
    (36,   "Segment",          None),
    (44,   "Spambase",         None),
    (287,  "WineQuality",      None),
]

for data_id, name, max_rows in openml_sources:
    try:
        d = fetch_openml_safe(
            data_id,
            retries=OPENML_RETRIES,
            base_delay=OPENML_BASE_DELAY
        )
        # Capture categorical cardinalities before prepare_tabular recodes
        cards = {}
        for col in d.data.columns:
            if hasattr(d.data[col], "cat"):
                cards[col] = len(d.data[col].cat.categories)
        add_dataset(name, d.data, d.target, max_rows=max_rows, category_cards=cards)
    except Exception as e:
        print(f"SKIP {name:<15} reason: {e}")
    time.sleep(5)

print(f"\n{len(datasets)} datasets loaded: {list(datasets.keys())}")

# ----------------------------------------------------------------------
# FIXED: classes_public is now hardcoded from public dataset documentation
# instead of computed from the loaded sample. This avoids a zero‑noise
# privacy leak under add/remove DP (if a rare class disappears from the
# sample, np.unique(y) would change). The numbers below must be verified
# against the official UCI/OpenML pages before final submission.
# ----------------------------------------------------------------------
PUBLIC_N_CLASSES = {
    "BreastCancer": 2,
    "PimaDiabetes": 2,
    "HeartHungarian": 2,
    "HeartStatlog": 2,
    "CreditG": 2,
    "AdultIncome": 2,
    "BloodTransfusion": 2,
    "MAGIC04": 2,
    "Banknote": 2,
    "CervicalCancer": 5,
    "CreditFraud": 2,
    "BankMarketing": 2,
    "Covertype": 7,
    "Ionosphere": 2,
    "Sonar": 2,
    "Wine": 3,
    "WisconsinOriginal": 2,
    "Waveform": 3,
    "Vehicle": 4,
    "Segment": 7,
    "Spambase": 2,
    "WineQuality": 7,
}

# Create the public classes mapping using the hardcoded counts.
# prepare_tabular() always encodes labels as 0..k-1 via pd.factorize,
# so np.arange(k) matches exactly.
classes_public = {
    ds_name: np.arange(PUBLIC_N_CLASSES[ds_name])
    for ds_name in datasets
    if ds_name in PUBLIC_N_CLASSES
}

# ---- Clusterability scoring ----
def dataset_clusterability(X_norm, y, seed, k=None):
    k = k or max(2, y.nunique())
    km = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X_norm)
    return silhouette_score(X_norm, km.labels_)

clusterability = {}
for ds_name, (X, y) in datasets.items():
    scaler = MinMaxScaler()
    X_n = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
    clusterability[ds_name] = dataset_clusterability(X_n, y, SEED)

print(pd.Series(clusterability).sort_values(ascending=False).to_string())

In [ ]:
# Cell 2b -- public feature bounds from UCI/OpenML documentation
# These are NOT derived from the current sample; they are prior knowledge.
# VERIFY column names against your actual `datasets[name][0].columns` output
# before trusting these -- a mismatched key silently falls to the wide
# (-1e3, 1e3) fallback, which flattens that column to near-zero after scaling.

PUBLIC_BOUNDS = {
    "BreastCancer": {
        "mean radius": (6.981, 28.11), "mean texture": (9.71, 39.28),
        "mean perimeter": (43.79, 188.5), "mean area": (143.5, 2501.0),
        "mean smoothness": (0.05263, 0.1634), "mean compactness": (0.01938, 0.3454),
        "mean concavity": (0.0, 0.4268), "mean concave points": (0.0, 0.2012),
        "mean symmetry": (0.106, 0.304), "mean fractal dimension": (0.04996, 0.09744),
        "radius error": (0.1115, 2.873), "texture error": (0.3602, 4.885),
        "perimeter error": (0.757, 21.98), "area error": (6.802, 542.2),
        "smoothness error": (0.001713, 0.03113), "compactness error": (0.002252, 0.1354),
        "concavity error": (0.0, 0.396), "concave points error": (0.0, 0.05279),
        "symmetry error": (0.007882, 0.07895), "fractal dimension error": (0.0008948, 0.02984),
        "worst radius": (7.93, 36.04), "worst texture": (12.02, 49.54),
        "worst perimeter": (50.41, 251.2), "worst area": (185.2, 4254.0),
        "worst smoothness": (0.07117, 0.2226), "worst compactness": (0.02729, 1.058),
        "worst concavity": (0.0, 1.252), "worst concave points": (0.0, 0.291),
        "worst symmetry": (0.1565, 0.6638), "worst fractal dimension": (0.05504, 0.2075),
    },
    "PimaDiabetes": {
        "preg": (0, 17), "plas": (0, 199), "pres": (0, 122), "skin": (0, 99),
        "insu": (0, 846), "mass": (0.0, 67.1), "pedi": (0.078, 2.42), "age": (21, 81),
    },
    "HeartHungarian": {
        "age": (28, 66), "trestbps": (94, 200), "chol": (126, 564),
        "thalach": (71, 202), "oldpeak": (0.0, 6.2),
    },
    "HeartStatlog": {
        "age": (29, 77), "trestbps": (94, 200), "chol": (126, 564),
        "thalach": (71, 202), "oldpeak": (0.0, 6.2),
    },

    "CreditG": {
        "duration": (4, 72),
        "credit_amount": (250, 18424),
        "installment_commitment": (1, 4),
        "residence_since": (1, 4),
        "age": (19, 75),
        "existing_credits": (1, 4),
        "num_dependents": (1, 2),
    },
    "AdultIncome": {
        "age": (17, 90), "fnlwgt": (12285, 1484705), "education-num": (1, 16),
        "capital-gain": (0, 99999), "capital-loss": (0, 4356), "hours-per-week": (1, 99),
    },
    "BloodTransfusion": {
        "V1": (2, 74), "V2": (1, 50), "V3": (250, 12500), "V4": (2, 98),
    },
    # ---- Fixed: MAGIC04 column names have trailing colons ----
    "MAGIC04": {
        "fLength:": (10.89, 317.98), "fWidth:": (5.29, 262.96), "fSize:": (0.4, 1028.0),
        "fConc:": (0.05, 0.99), "fConc1:": (0.0, 0.94), "fAsym:": (-24.68, 24.68),
        "fM3Long:": (-331.0, 331.0), "fM3Trans:": (-331.0, 331.0),
        "fAlpha:": (0.0, 90.0), "fDist:": (20.0, 500.0),
    },
    "Banknote": {
        "V1": (-7.0421, 6.8248), "V2": (-13.7731, 12.9516),
        "V3": (-5.2861, 17.9274), "V4": (-8.5482, 2.4495),
    },
    "Ionosphere": {
        **{f"a{str(i).zfill(2)}": (-1.0, 1.0) for i in range(1, 35)},
        **{f"V{i}": (-1.0, 1.0) for i in range(1, 35)},
    },
    "Sonar": {
        **{f"V{i}": (0.0, 1.0) for i in range(1, 61)},
        **{f"attribute_{i}": (0.0, 1.0) for i in range(1, 61)},
    },
    # ---- Fixed: Wine uses lowercase after underscore in several names ----
    "Wine": {
        "Alcohol": (11.03, 14.83), "Malic_acid": (0.74, 5.80), "Ash": (1.36, 3.23),
        "Alcalinity_of_ash": (10.6, 30.0), "Magnesium": (70, 162),
        "Total_phenols": (0.98, 3.88), "Flavanoids": (0.34, 5.08),
        "Nonflavanoid_phenols": (0.13, 0.66), "Proanthocyanins": (0.41, 3.58),
        "Color_intensity": (1.28, 13.0), "Hue": (0.48, 1.71),
        "OD280%2FOD315_of_diluted_wines": (1.27, 4.00), "Proline": (278, 1680),
    },
    "WisconsinOriginal": {
        "Clump_Thickness": (1, 10), "Cell_Size_Uniformity": (1, 10),
        "Cell_Shape_Uniformity": (1, 10), "Marginal_Adhesion": (1, 10),
        "Single_Epi_Cell_Size": (1, 10), "Bare_Nuclei": (1, 10),
        "Bland_Chromatin": (1, 10), "Normal_Nucleoli": (1, 10), "Mitoses": (1, 10),
    },
    # ---- Fixed: Waveform actually has 40 columns named x1..x40 ----
    "Waveform": {f"x{i}": (-6.0, 15.0) for i in range(1, 41)},
    # ---- Fixed: Vehicle uses descriptive names (still loose placeholder) ----
    "Vehicle": {
        "COMPACTNESS": (0.0, 1000.0), "CIRCULARITY": (0.0, 1000.0),
        "DISTANCE_CIRCULARITY": (0.0, 1000.0), "RADIUS_RATIO": (0.0, 1000.0),
        "PR.AXIS_ASPECT_RATIO": (0.0, 1000.0), "MAX.LENGTH_ASPECT_RATIO": (0.0, 1000.0),
        "SCATTER_RATIO": (0.0, 1000.0), "ELONGATEDNESS": (0.0, 1000.0),
        "PR.AXIS_RECTANGULARITY": (0.0, 1000.0), "MAX.LENGTH_RECTANGULARITY": (0.0, 1000.0),
        "SCALED_VARIANCE_MAJOR": (0.0, 1000.0), "SCALED_VARIANCE_MINOR": (0.0, 1000.0),
        "SCALED_RADIUS_OF_GYRATION": (0.0, 1000.0), "SKEWNESS_ABOUT_MAJOR": (0.0, 1000.0),
        "SKEWNESS_ABOUT_MINOR": (0.0, 1000.0), "KURTOSIS_ABOUT_MAJOR": (0.0, 1000.0),
        "KURTOSIS_ABOUT_MINOR": (0.0, 1000.0), "HOLLOWS_RATIO": (0.0, 1000.0),
    },
    # ---- Fixed: Segment uses descriptive names (still loose placeholder) ----
    "Segment": {
        "region-centroid-col": (-300.0, 300.0), "region-centroid-row": (-300.0, 300.0),
        "short-line-density-5": (-300.0, 300.0), "short-line-density-2": (-300.0, 300.0),
        "vedge-mean": (-300.0, 300.0), "vegde-sd": (-300.0, 300.0),
        "hedge-mean": (-300.0, 300.0), "hedge-sd": (-300.0, 300.0),
        "intensity-mean": (-300.0, 300.0), "rawred-mean": (-300.0, 300.0),
        "rawblue-mean": (-300.0, 300.0), "rawgreen-mean": (-300.0, 300.0),
        "exred-mean": (-300.0, 300.0), "exblue-mean": (-300.0, 300.0),
        "exgreen-mean": (-300.0, 300.0), "value-mean": (-300.0, 300.0),
        "saturation-mean": (-300.0, 300.0), "hue-mean": (-300.0, 300.0),
    },
    # Spambase and WineQuality are handled dynamically after loading (see below)
    "CervicalCancer": {},
    "CreditFraud": {},
    "BankMarketing": {},
    "Covertype": {},
}

def public_bounds_for(ds_name, X):
    d = X.shape[1]
    lo = np.zeros(d)
    hi = np.ones(d)
    doc = PUBLIC_BOUNDS.get(ds_name, {})
    cat_cards = CATEGORY_CARDINALITY.get(ds_name, {})
    for j, col in enumerate(X.columns):
        if col in doc:
            lo[j], hi[j] = doc[col]
        elif col in cat_cards:
            lo[j], hi[j] = 0, cat_cards[col] - 1
        else:
            lo[j], hi[j] = -1e3, 1e3
    return lo, hi

# Coverage check -- run this after building `datasets`, before Cell 7.
# Flags any column that will silently hit the wide fallback.
for ds_name in PUBLIC_BOUNDS:
    if ds_name not in datasets:
        continue
    X, _ = datasets[ds_name]
    doc = PUBLIC_BOUNDS[ds_name]
    cat = CATEGORY_CARDINALITY.get(ds_name, {})
    missing = [c for c in X.columns if c not in doc and c not in cat]
    if missing:
        print(f"UNCOVERED in {ds_name} (will use -1e3..1e3 fallback): {missing}")

# Dynamic bounds for Spambase and WineQuality (runs after coverage-check loop)
if "Spambase" in datasets:
    X_sb, _ = datasets["Spambase"]
    PUBLIC_BOUNDS["Spambase"] = {
        c: (0.0, 100.0) if c.startswith(("word_freq_", "char_freq_"))
        else (1.0, 1000.0) if c == "capital_run_length_average"
        else (1.0, 10000.0) if c == "capital_run_length_longest"
        else (1.0, 20000.0)   # capital_run_length_total
        for c in X_sb.columns
    }

if "WineQuality" in datasets:
    PUBLIC_BOUNDS["WineQuality"] = {
        "fixed.acidity": (4.6, 15.9), "volatile.acidity": (0.08, 1.58),
        "citric.acid": (0.0, 1.66), "residual.sugar": (0.6, 65.8),
        "chlorides": (0.009, 0.611), "free.sulfur.dioxide": (1, 289),
        "total.sulfur.dioxide": (6, 440), "density": (0.98, 1.039),
        "pH": (2.72, 4.01), "sulphates": (0.22, 2.0), "alcohol": (8.0, 14.9),
    }

In [ ]:
# Cell 3: DP-consistent granular-ball construction — core private mechanisms.
#
# Two private sub-mechanisms, both standard and citable:
#  (a) Private purity check -> Laplace mechanism on class counts.
#      Sensitivity = 1 (adding/removing one record changes a count by <= 1).
#  (b) Private split selection -> Exponential Mechanism over candidate
#      (feature, threshold) pairs, scored by the majority-count split score
#      (see Cell 3d), sensitivity = 1.
#
# Composition: fixed max_depth D is chosen BEFORE looking at data (public
# hyperparameter). At each depth, all nodes are disjoint partitions of the
# training set -> parallel composition -> one (eps_count + eps_split) charge
# per depth level, not per node. Total granulation cost = D * (eps_count + eps_split).
# =============================================================================
# NEIGHBOURING RELATION: ADD/REMOVE (one record added or removed).
# Under this definition, the sensitivity of any count is 1.
# All Laplace mechanisms in this notebook assume ADD/REMOVE.
# If you switch to SUBSTITUTE-ONE, you must change sensitivities to 2 where
# appropriate and re-derive budgets.
# =============================================================================
def laplace_mech(true_value, sensitivity, eps, rng):
    if eps <= 0:
        return true_value
    scale = sensitivity / eps
    return true_value + rng.laplace(0, scale)

def exponential_mechanism(candidates, scores, sensitivity, eps, rng):
    """candidates: list of items; scores: matching list of utility scores
    (higher = better). Returns one sampled candidate."""
    if eps <= 0 or len(candidates) == 0:
        return candidates[rng.integers(len(candidates))]
    scores = np.asarray(scores, dtype=float)
    logits = (eps * scores) / (2.0 * sensitivity)
    logits -= logits.max()
    probs = np.exp(logits)
    probs /= probs.sum()
    idx = rng.choice(len(candidates), p=probs)
    return candidates[idx]

print("Laplace mechanism + Exponential mechanism ready. "
      "The granular-ball builder itself is defined in Cell 3d "
      "(dp_generate_granular_balls_with_radius_fixed), the only version now kept.")

In [ ]:
# Cell 3c — shared helpers used by the granular-ball builder and by direct
# ball classification.
#
# REMOVED: dp_release_ball_radius() and dp_release_ball_center() — these were
# dead code. The active mechanism (Cell 3d) computes center/radius as free
# public post-processing of the already-released split thresholds (box
# midpoint / box half-diagonal), so no per-ball Laplace/Gaussian release of
# center or radius ever actually runs in this notebook.
from scipy.spatial.distance import cdist

def dp_release_ball_counts(balls, eps_count_release, rng):
    """How many synthetic records to emit per ball. Sensitivity=1 (one record
    changes exactly one ball's true size by at most 1), pure-eps Laplace."""
    sizes = []
    for ball in balls:
        true_n = len(ball["indices"])
        noisy_n = laplace_mech(true_n, sensitivity=1.0, eps=eps_count_release, rng=rng)
        sizes.append(max(0, int(round(noisy_n))))
    return sizes


def dp_gbc_classify_predict(balls, X_te_scaled):
    """
    Classify directly from DP-released ball geometry -- no synthetic feature
    sampling. Post-processing of released (center, noisy_class_counts):
    zero additional budget.
    """
    centers = np.array([b["center"] for b in balls])
    X_te_arr = X_te_scaled.values if hasattr(X_te_scaled, "values") else np.asarray(X_te_scaled)
    dists = cdist(X_te_arr, centers)
    nearest = np.argmin(dists, axis=1)
    ball_preds = []
    for b in balls:
        counts = np.clip(b["noisy_class_counts"], 0, None)
        ball_preds.append(b["classes"][np.argmax(counts)] if counts.sum() > 0 else b["classes"][0])
    return np.array(ball_preds)[nearest]

In [ ]:
# Cell 3d (CORRECTED & FIXED): majority-count split score, sensitivity PROVABLY = 1.
# Gini gain (previous version) does not have a small, n-/balance-independent
# sensitivity bound -- see Fletcher & Islam 2019 survey noting Friedman &
# Schuster preferred the count-based "Max" operator specifically for its low,
# provable sensitivity over impurity-based scores: adding/removing one record
# changes exactly one class's count on exactly one side of a candidate split
# by exactly 1, so |Δscore| <= 1 always.
#
# NOTE: this cell now defines the ONLY granular-ball builder used anywhere in
# the notebook (dp_generate_granular_balls_with_radius_fixed). The earlier
# non-radius "fixed" variant was dead code -- nothing ever called it -- and
# has been removed, along with the stale-definition sanity check that existed
# only to guard against it.

def majority_count_score(y_node, vals, t):
    left_mask = vals <= t
    y_left, y_right = y_node[left_mask], y_node[~left_mask]
    if len(y_left) == 0 or len(y_right) == 0:
        return 0.0
    left_counts = np.bincount(y_left.astype(int))
    right_counts = np.bincount(y_right.astype(int))
    return float(left_counts.max() + right_counts.max())

def dp_generate_granular_balls_with_radius_fixed(X, y, max_depth, eps_count_per_level,
                                                 eps_split_per_level, eps_radius_per_level,
                                                 eps_center_per_level=0.1,
                                                 min_size=10, n_candidate_thresholds=8, rng=None,
                                                 all_classes=None):
    rng = rng or np.random.default_rng(SEED)
    X_arr = X.values if hasattr(X, "values") else np.asarray(X)
    y_arr = np.asarray(y)
    global_classes = all_classes if all_classes is not None else np.unique(y_arr)
    n_features = X_arr.shape[1]
    balls = []
    root_lo = np.zeros(n_features)
    root_hi = np.ones(n_features)
    queue = [(np.arange(len(X_arr)), 0, root_lo, root_hi)]

    while queue:
        indices, depth, box_lo, box_hi = queue.pop(0)
        X_node, y_node = X_arr[indices], y_arr[indices]

        counts_full = np.array([np.sum(y_node == c) for c in global_classes])
        noisy_counts = np.array([laplace_mech(c, sensitivity=1.0, eps=eps_count_per_level, rng=rng) for c in counts_full])
        noisy_counts = np.clip(noisy_counts, 0, None) + 1.0
        noisy_purity = noisy_counts.max() / max(noisy_counts.sum(), 1e-6)

        stop = (noisy_purity >= 0.90) or (noisy_counts.sum() < min_size) or (depth >= max_depth)

        if stop:
            center = (box_lo + box_hi) / 2.0
            radius = np.linalg.norm(box_hi - box_lo) / 2.0
            balls.append({
                "indices": indices, "noisy_purity": noisy_purity,
                "center": center, "radius": radius,
                "box_lo": box_lo.copy(), "box_hi": box_hi.copy(),
                "n": len(indices), "is_boundary": noisy_purity < 0.90,
                "classes": global_classes, "noisy_class_counts": np.clip(noisy_counts, 0, None)
            })
            continue

        candidates, scores = [], []
        for f in range(n_features):
            lo_f, hi_f = box_lo[f], box_hi[f]
            if hi_f - lo_f < 1e-9:
                continue
            vals = X_node[:, f]
            thresholds = np.linspace(lo_f, hi_f, n_candidate_thresholds + 2)[1:-1]
            for t in thresholds:
                left = vals <= t
                candidates.append((f, t))
                if left.sum() < 2 or (~left).sum() < 2:
                    scores.append(0.0)
                else:
                    scores.append(majority_count_score(y_node, vals, t))

        if not candidates:
            center = (box_lo + box_hi) / 2.0
            radius = np.linalg.norm(box_hi - box_lo) / 2.0
            balls.append({
                "indices": indices, "noisy_purity": noisy_purity,
                "center": center, "radius": radius,
                "box_lo": box_lo.copy(), "box_hi": box_hi.copy(),
                "n": len(indices), "is_boundary": noisy_purity < 0.90,
                "classes": global_classes, "noisy_class_counts": np.clip(noisy_counts, 0, None)
            })
            continue

        chosen_f, chosen_t = exponential_mechanism(candidates, scores, 1.0, eps_split_per_level, rng)
        left_mask = X_node[:, chosen_f] <= chosen_t

        left_lo, left_hi = box_lo.copy(), box_hi.copy()
        left_hi[chosen_f] = np.clip(min(left_hi[chosen_f], chosen_t), box_lo[chosen_f], box_hi[chosen_f])
        right_lo, right_hi = box_lo.copy(), box_hi.copy()
        right_lo[chosen_f] = np.clip(max(right_lo[chosen_f], chosen_t), box_lo[chosen_f], box_hi[chosen_f])

        queue.append((indices[left_mask], depth + 1, left_lo, left_hi))
        queue.append((indices[~left_mask], depth + 1, right_lo, right_hi))

    return balls

# Single canonical name used by every other cell in the notebook.
dp_generate_granular_balls_with_radius = dp_generate_granular_balls_with_radius_fixed

print("FIX APPLIED: majority-count split score, global class counts, public box radii, "
      "noised stopping, no empty-child guard, box-scoped thresholds, clipped child boxes. "
      "Dead/stale builder variants removed — one canonical implementation remains.")

In [ ]:
# Cell 4 (FIXED): DP-GBC synthetic release.
# Replaces apply_region_noise (data-dependent Gaussian noise) with:
#   (1) DP-release a count per ball
#   (2) sample synthetic points from the ball's released statistics
# Sampling from released stats is pure post-processing, zero extra budget.
#
# FIX B: MinMax scaling is replaced by one-time DP-released feature bounds.
# Bounds are released once per dataset, outside the seed loop, and are public afterward.
# FIX C: Synthetic points are sampled uniformly from each ball's box, not an L2 sphere.
# FIX D: Label sampling uses 'classes' + 'noisy_class_counts' (not 'class_counts').
# FIX E: Global safeguard handles any number of classes (not just binary).
# NEW: sampling_mode parameter ("uniform" default, "gaussian" optional).


class PrivacyLedger:
    """Tracks spent (ε, δ) privacy budget using sequential and parallel composition."""
    def __init__(self):
        self.eps_spent = 0.0
        self.delta_spent = 0.0
        self.records = []

    def spend_sequential(self, eps, delta, note=""):
        self.eps_spent += eps
        self.delta_spent += delta
        self.records.append((eps, delta, note))

    def spend_sequential_charged_once(self, eps, delta, n_branches, note=""):
        """
        NOT true parallel composition.
        This method exists to record a sequential charge where the caller has
        already applied parallel composition across disjoint branches and is
        passing the effective per-block (eps, delta) once. It calls
        spend_sequential to keep the ledger strict and conservative.
        """
        self.spend_sequential(eps, delta, note=f"parallel({n_branches}) charged once: {note}")

    @property
    def total_eps(self):
        return self.eps_spent

    @property
    def total_delta(self):
        return self.delta_spent


def scale_with_public_bounds(X, lo, hi):
    """Clip to DP-released bounds and scale to [0,1]. Public, no extra budget."""
    X_arr = X.values if hasattr(X, "values") else np.asarray(X)
    span = np.maximum(hi - lo, 1e-6)
    cols = X.columns if hasattr(X, "columns") else None
    idx = X.index if hasattr(X, "index") else None
    return pd.DataFrame(np.clip((X_arr - lo) / span, 0, 1), columns=cols, index=idx)


def dp_sample_synthetic_records(balls, sizes, rng, d, sampling_mode="uniform"):
    """Generate synthetic records from each ball's released box geometry.

    sampling_mode:
      "uniform"  (default, ORIGINAL mechanism, unchanged) — sample uniformly
                 within the box. Costs zero extra privacy budget.
      "gaussian" (NEW, ablation) — sample from a diagonal Gaussian centered
                 at the box midpoint, std = box_width / 6, then clipped back
                 into the box. Mean/std are a DETERMINISTIC function of the
                 already-released box_lo/box_hi (public post-processing of
                 already-released quantities) — NOT a new noisy release of
                 the true per-ball mean/covariance. Costs exactly the same
                 privacy budget as "uniform": zero extra.
    """
    assert sampling_mode in ("uniform", "gaussian"), sampling_mode
    X_synth, y_synth = [], []

    for ball, n_out in zip(balls, sizes):
        if n_out <= 0:
            continue

        box_lo = ball.get("box_lo", np.zeros(d))
        box_hi = ball.get("box_hi", np.ones(d))
        box_hi = np.maximum(box_hi, box_lo)  # guarantee high >= low

        if sampling_mode == "uniform":
            X_new = rng.uniform(low=box_lo, high=box_hi, size=(int(n_out), d))
        else:  # "gaussian"
            center = (box_lo + box_hi) / 2.0
            width = np.maximum(box_hi - box_lo, 1e-6)
            std = width / 6.0  # ~99.7% mass inside the box pre-clipping
            X_new = rng.normal(loc=center, scale=std, size=(int(n_out), d))
            X_new = np.clip(X_new, box_lo, box_hi)

        X_synth.append(X_new)

        # ---- FIX D: sample labels using DP-released classes + noisy counts ----
        classes = ball.get("classes")
        noisy_counts = ball.get("noisy_class_counts")
        if classes is not None and noisy_counts is not None:
            counts = np.clip(np.asarray(noisy_counts, dtype=float), 0, None)
            total = counts.sum()
            if total > 0:
                probs = counts / total
                y_new = rng.choice(classes, size=int(n_out), p=probs)   # actual labels
            else:
                y_new = np.full(int(n_out), classes[0])
        else:
            y_new = np.zeros(int(n_out), dtype=int)
        # ----------------------------------------------------------------

        y_synth.extend(y_new)

    # NOTE (Fix 6): The forced-class injection safeguard has been removed.
    # Instead, class collapse will be reported as a diagnostic metric.

    if len(X_synth) == 0:
        return np.empty((0, d)), np.empty(0, dtype=int)

    X_out = np.vstack(X_synth)
    y_out = np.asarray(y_synth, dtype=int)
    return X_out, y_out


def run_dp_gbc_pipeline(
    X, y, seed, lo, hi, max_depth=4,
    eps_count_per_level=0.5,
    eps_split_per_level=0.5,
    eps_center_per_level=0.1,
    eps_radius_per_level=0.1,          # radius must be released for synthetic sampling
    eps_count_release=0.5,             # budget for per-ball synthetic size
    delta=1e-5,
    all_classes=None,
    precomputed_split=None,            # NEW: optional fixed (X_tr, X_te, y_tr, y_te)
    sampling_mode="uniform",           # <-- NEW, default preserves old behavior
):
    rng = np.random.default_rng(seed)
    ledger = PrivacyLedger()

    # Train/test split: use precomputed_split if provided, else split internally
    if precomputed_split is not None:
        X_tr, X_te, y_tr, y_te = precomputed_split
    else:
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.3, random_state=seed, stratify=y
        )

    # FIX B: public-bounds scaling (no private MinMax)
    X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
    X_te_n = scale_with_public_bounds(X_te, lo, hi)

    # DP granular balls, now with radius + class counts
    balls = dp_generate_granular_balls_with_radius(
        X_tr_n, y_tr.values,
        max_depth=max_depth,
        eps_count_per_level=eps_count_per_level,
        eps_split_per_level=eps_split_per_level,
        eps_radius_per_level=eps_radius_per_level,
        eps_center_per_level=eps_center_per_level,
        rng=rng,
        all_classes=all_classes
    )

    # Granulation budget: pure-eps per depth, parallel across siblings
    # FIX 3: Remove center and radius from the budget. Reallocate to count/split.
    per_level = (eps_count_per_level + eps_split_per_level)
    for depth in range(max_depth):
        ledger.spend_sequential_charged_once(
            per_level, delta=0.0, n_branches=len(balls),          # was delta=delta
            note=f"granulation depth {depth+1} (pure-eps)"
        )

    # DP-release how many synthetic rows each ball should emit.
    # Sensitivity = 1 (one record changes one ball size by at most 1).
    ledger.spend_sequential(
        eps_count_release, 0.0,
        note="per-ball synthetic-size release (pure-eps)"
    )
    sizes = dp_release_ball_counts(balls, eps_count_release, rng)

    # Generate synthetic training data from released ball statistics.
    # This is post-processing — zero additional privacy cost.
    X_synth, y_synth = dp_sample_synthetic_records(
        balls, sizes, rng, d=X_tr_n.shape[1],
        sampling_mode=sampling_mode          # <-- pass through
    )

    # Test set stays as real holdout, exactly like the oracle/flat baselines.
    X_tr_out = pd.DataFrame(X_synth, columns=X_tr_n.columns)
    y_tr_out = pd.Series(y_synth)

    # Compatibility: return a boundary rate.
    boundary_rate = (
        float(np.mean([b["is_boundary"] for b in balls]))
        if balls else 0.0
    )

    return X_tr_out, X_te_n, y_tr_out, y_te, ledger, boundary_rate, balls

In [ ]:
# Cell 4c (UPDATED — Fix 1)
# DP-LocalClip is now an alias for DP-GBC. No separate mechanism.
# Both pipelines produce identical DP synthetic releases.
# This cell only defines the alias; PrivacyLedger is assumed to be defined elsewhere
# (e.g., Cell 1 or wherever it was originally defined).

def run_dp_localclip_pipeline(
    X, y, seed, lo, hi, max_depth=3,
    eps_count_per_level=0.1,
    eps_split_per_level=0.1,
    eps_radius_per_level=0.1,
    eps_center_per_level=0.1,
    eps_noise=1.5,
    delta=1e-5,
    all_classes=None,
    precomputed_split=None,
    sampling_mode="uniform",          # <-- NEW
):
    """
    ALIAS, not a separate mechanism: verified line-for-line identical to
    run_dp_gbc_pipeline (same granular-ball generation, same per-ball count
    release, same box-uniform synthetic sampling). Kept only so existing call
    sites (Cells 8, 14, 16, 23, 25) don't need editing. It forwards to
    run_dp_gbc_pipeline so "DP-GBC" and "DP-LocalClip" numbers can never
    silently diverge into two different mechanisms again.
    In the paper: ONE mechanism ("DP-GBC synthetic release"), evaluated with
    different downstream heads (direct-ball / +LR / +MLP) -- never framed as
    two competing privacy mechanisms.
    """
    X_tr_out, X_te_n, y_tr_out, y_te, ledger, boundary_rate, balls = run_dp_gbc_pipeline(
        X, y, seed, lo, hi,
        max_depth=max_depth,
        eps_count_per_level=eps_count_per_level,
        eps_split_per_level=eps_split_per_level,
        eps_center_per_level=eps_center_per_level,
        eps_radius_per_level=eps_radius_per_level,
        eps_count_release=eps_noise,
        delta=delta,
        all_classes=all_classes,
        precomputed_split=precomputed_split,
        sampling_mode=sampling_mode,          # <-- pass through
    )
    mean_radius = float(np.mean([b["radius"] for b in balls])) if balls else float("nan")
    return X_tr_out, X_te_n, y_tr_out, y_te, ledger, mean_radius

In [ ]:
# Cell 5: Fair utility evaluation. Report accuracy AND balanced accuracy AND
# F1/AUPRC on the minority class, so a "predict majority always" collapse is
# visible and not hidden behind a misleadingly high accuracy number.

from sklearn.neural_network import MLPClassifier
from sklearn.utils import resample

def evaluate_utility(X_tr, y_tr, X_te, y_te):
    model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    probs = model.predict_proba(X_te)[:, 1] if len(np.unique(y_tr)) == 2 else None

    # MLPClassifier has no class_weight/sample_weight support (confirmed via
    # scikit-learn GitHub issue #9113 and discussion #23544), so we manually
    # oversample minority classes before fitting -- otherwise the MLP trains
    # unbalanced while LogisticRegression above trains balanced, making the
    # two-architecture comparison apples-to-oranges.
    df_tr = X_tr.copy()
    y_tr_arr = y_tr.values if hasattr(y_tr, "values") else np.asarray(y_tr)
    df_tr["_y"] = y_tr_arr
    parts = [df_tr[df_tr["_y"] == c] for c in np.unique(y_tr_arr)]
    max_n = max(len(p) for p in parts)
    balanced_tr = pd.concat(
        [resample(p, replace=True, n_samples=max_n, random_state=SEED) for p in parts]
    )
    model_mlp = MLPClassifier(hidden_layer_sizes=(16,), max_iter=500, random_state=SEED)
    model_mlp.fit(balanced_tr.drop(columns="_y"), balanced_tr["_y"])
    preds_mlp = model_mlp.predict(X_te)

    out = {
        "accuracy": accuracy_score(y_te, preds),
        "balanced_accuracy": balanced_accuracy_score(y_te, preds),
        "mlp_balanced_accuracy": balanced_accuracy_score(y_te, preds_mlp),
        "f1_minority": f1_score(y_te, preds, average="binary",
                                 pos_label=pd.Series(y_te).value_counts().idxmin())
                        if len(np.unique(y_te)) == 2 else np.nan,
    }
    if probs is not None:
        out["auprc"] = average_precision_score(y_te, probs)
    return out, model

In [ ]:
# Cell 6: Attacks, PLUS a sanity check that the attack actually works when
# there IS something to detect. Without this control, an MIA_AUC near 0.5
# under DP is uninterpretable -- you can't tell "well protected" apart from
# "the attack doesn't work."

def membership_inference_attack(model, X_tr, y_tr, X_te, y_te):
    conf_tr = np.max(model.predict_proba(X_tr), axis=1)
    conf_te = np.max(model.predict_proba(X_te), axis=1)
    scores = np.concatenate([conf_tr, conf_te])
    labels = np.concatenate([np.ones(len(conf_tr)), np.zeros(len(conf_te))])
    try:
        return roc_auc_score(labels, scores)
    except Exception:
        return 0.5

def positive_control_mia(X_tr, y_tr, X_te, y_te, seed):
    """Deliberately overfit an unregularized high-capacity RF on RAW
    (non-private) data. If MIA_AUC here isn't well above 0.5, your attack
    implementation itself is the problem, not the privacy mechanism."""
    overfit_model = RandomForestClassifier(
        n_estimators=200, max_depth=None, min_samples_leaf=1, random_state=seed
    )
    overfit_model.fit(X_tr, y_tr)
    return membership_inference_attack(overfit_model, X_tr, y_tr, X_te, y_te)

def membership_inference_attack_rf(X_tr, y_tr, X_te, y_te, seed):
    """Confidence-gap membership inference attack (Yeom et al. 2018 style):
    trains a classifier directly on the target's own training set and
    measures whether its confidence differs between members (X_tr) and
    non-members (X_te). NOT a Shokri et al. (2017) shadow-model attack --
    no separate model trained on disjoint data is used here."""
    # Confidence-gap model: RF trained on train set
    model = RandomForestClassifier(
        n_estimators=100, max_depth=6,
        class_weight="balanced", random_state=seed
    )
    model.fit(X_tr, y_tr)
    # Features: confidence scores from the model
    conf_tr = np.max(model.predict_proba(X_tr), axis=1)
    conf_te = np.max(model.predict_proba(X_te), axis=1)
    scores = np.concatenate([conf_tr, conf_te])
    labels = np.concatenate([np.ones(len(conf_tr)), np.zeros(len(conf_te))])
    try:
        auc = roc_auc_score(labels, scores)
    except Exception:
        auc = 0.5
    return auc, model

In [ ]:
# Cell 6b (NEW): Attribute Inference Attack, restored from your original design
# but fixed: uses a fixed seed correctly, class_weight balanced, and reports
# lift over the trivial majority-class baseline rather than raw accuracy alone.

from sklearn.preprocessing import KBinsDiscretizer

def attribute_inference_attack(X_tr_attack, X_te_attack, sens_tr, sens_te, seed):
    sens_tr = np.nan_to_num(np.asarray(sens_tr).ravel(), nan=0.0)
    sens_te = np.nan_to_num(np.asarray(sens_te).ravel(), nan=0.0)

    if len(np.unique(sens_tr)) > 10:  # continuous -> discretize into 3 bins
        disc = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile")
        sens_tr = disc.fit_transform(sens_tr.reshape(-1, 1)).ravel().astype(int)
        sens_te = disc.transform(sens_te.reshape(-1, 1)).ravel().astype(int)
    else:
        sens_tr, sens_te = sens_tr.astype(int), sens_te.astype(int)

    if len(np.unique(sens_tr)) < 2:
        print(f"    [AIA Note] Sensitive attribute collapsed to 1 bin due to noise; returning 0.0 lift by construction.")
        majority = np.bincount(sens_tr).argmax()
        preds = np.full_like(sens_te, majority)
        baseline = pd.Series(sens_te).value_counts(normalize=True).max()
        return accuracy_score(sens_te, preds) - baseline

    attacker = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=seed)
    attacker.fit(X_tr_attack, sens_tr)
    preds = attacker.predict(X_te_attack)
    baseline = pd.Series(sens_te).value_counts(normalize=True).max()
    return accuracy_score(sens_te, preds) - baseline  # "lift" over guessing majority sensitive value

def run_aia(X_tr_noisy, y_tr, X_te_noisy, y_te, model, sens_col_idx=0, seed=SEED):
    """Attack tries to infer feature[sens_col_idx] from the OTHER features + model outputs."""
    non_sens_cols = [c for i, c in enumerate(X_tr_noisy.columns) if i != sens_col_idx]
    X_tr_attack = np.hstack([X_tr_noisy[non_sens_cols].values, model.predict_proba(X_tr_noisy)])
    X_te_attack = np.hstack([X_te_noisy[non_sens_cols].values, model.predict_proba(X_te_noisy)])
    sens_tr = X_tr_noisy.iloc[:, sens_col_idx].values
    sens_te = X_te_noisy.iloc[:, sens_col_idx].values
    return attribute_inference_attack(X_tr_attack, X_te_attack, sens_tr, sens_te, seed)

In [ ]:
# Cell 6c (NEW): Correct MIA protocol for SYNTHETIC-data conditions.
# Per van Breugel et al. (AISTATS 2023) and TAPAS (Houssiau et al. 2022):
# membership is a property of REAL records. The attacker may only ever query
# real candidate points -- never synthetic rows, which have no 1:1 mapping
# to any actual individual and are not a valid member/non-member proxy.
def membership_inference_attack_rf_synth(X_tr_synth, y_tr_synth,
                                          X_real_members, y_real_members,
                                          X_real_nonmembers, y_real_nonmembers,
                                          seed):
    shadow = RandomForestClassifier(
        n_estimators=100, max_depth=6,
        class_weight="balanced", random_state=seed
    )
    shadow.fit(X_tr_synth, y_tr_synth)
    conf_members = np.max(shadow.predict_proba(X_real_members), axis=1)
    conf_nonmembers = np.max(shadow.predict_proba(X_real_nonmembers), axis=1)
    scores = np.concatenate([conf_members, conf_nonmembers])
    labels = np.concatenate([np.ones(len(conf_members)), np.zeros(len(conf_nonmembers))])
    try:
        return roc_auc_score(labels, scores), shadow
    except Exception:
        return 0.5, shadow

In [ ]:
# Cell 7 (REPLACE): flat-LR, flat-RF ("dp_forest" naive baseline), DP-GBC (direct & synthetic+LR).
# All conditions spend EXACTLY the same TOTAL_EPS. All datasets. All attacks.
# This is your main results table -- the one a reviewer will scrutinize hardest.

TOTAL_EPS = 3.0
SEEDS = [42, 123, 456, 789, 101]

NUMERIC_DATASETS = [
    "BreastCancer",
    "PimaDiabetes",
    "BloodTransfusion",
    "MAGIC04",
    "Banknote",
    "Ionosphere",
    "Sonar",
    "Wine",
    "WisconsinOriginal",
    "Waveform",
    "Vehicle",
    "Segment",
    "Spambase",
    "WineQuality",
]

# Hard-fail assertion: no numeric column may ever fall back silently again
uncovered_in_numeric = {}
for ds_name in NUMERIC_DATASETS:
    if ds_name not in datasets:
        continue
    X, _ = datasets[ds_name]
    doc = PUBLIC_BOUNDS.get(ds_name, {})
    cat = CATEGORY_CARDINALITY.get(ds_name, {})
    missing = [c for c in X.columns if c not in doc and c not in cat]
    if missing:
        uncovered_in_numeric[ds_name] = missing

assert not uncovered_in_numeric, (
    f"Uncovered columns in NUMERIC_DATASETS -- fix PUBLIC_BOUNDS keys before running "
    f"Cell 7: {uncovered_in_numeric}"
)
print("Coverage check passed: every NUMERIC_DATASETS column has a declared public bound.")

# Add a mapping of an actual sensitive/interesting column per dataset, e.g.:
SENSITIVE_COL = {
    "AdultIncome": "age",
    "BreastCancer": "mean radius",
    "PimaDiabetes": "age",     # OpenML id=37 "diabetes" uses lowercase "age", not "Age"
    "HeartHungarian": "age",
    "CreditG": "age",
}

def resolve_sens_col_idx(X, ds_name):
    """Case-insensitive lookup with a safe fallback to column 0, so a mismatched
    name warns instead of crashing the whole run."""
    target = SENSITIVE_COL.get(ds_name)
    if target is None:
        return 0
    cols_lower = [c.lower() for c in X.columns]
    if target.lower() in cols_lower:
        return cols_lower.index(target.lower())
    print(f"WARNING: sensitive column '{target}' not found for {ds_name} "
          f"(available: {list(X.columns)}) -- falling back to column 0")
    return 0

def make_flat_noised(X_n, eps, rng):
    d = X_n.shape[1]
    l2_sensitivity = np.sqrt(d)      # switch to Gaussian: L2 sensitivity sqrt(d) << L1 sensitivity d
    delta = 1e-5
    sigma = l2_sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / eps
    return pd.DataFrame(np.clip(X_n.values + rng.normal(0, sigma, X_n.shape), 0, 1),
                         columns=X_n.columns, index=X_n.index)

# --- Quick sanity check: do the sensitive column names actually exist? ---
print("Sanity check: sensitive column names in each dataset:")
for ds_name in ["AdultIncome", "BreastCancer", "PimaDiabetes", "HeartHungarian", "CreditG"]:
    if ds_name in datasets:
        X, _ = datasets[ds_name]
        print(f"  {ds_name} -> {list(X.columns)}")
    else:
        print(f"  {ds_name} -> dataset not loaded")

# ---------- CORRECTED evaluate_utility (fixes AUPRC) ----------
from sklearn.metrics import balanced_accuracy_score, average_precision_score

def evaluate_utility(X_tr, y_tr, X_te, y_te):
    """Train a logistic regression and return balanced accuracy and AUPRC."""
    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    bal_acc = balanced_accuracy_score(y_te, preds)
    out = {"balanced_accuracy": bal_acc, "auprc": np.nan}
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_te)
        if probs is not None:
            # binary case: use positive class scores
            if len(np.unique(y_te)) == 2:
                out["auprc"] = average_precision_score(y_te, probs[:, 1])
            else:
                out["auprc"] = average_precision_score(y_te, probs)  # multi-class, average='macro' by default
    return out, model

# ---------- NEW: Wrapper for DP‑GBC to expose predict_proba/predict ----------
from scipy.spatial.distance import cdist

class BallModelWrapper:
    """Exposes predict_proba/predict from a DP-GBC ball list so run_aia /
    membership_inference_attack can be called on Condition C directly."""
    def __init__(self, balls):
        self.balls = balls
        self.centers = np.array([b["center"] for b in balls])
        self.classes_ = np.unique(np.concatenate([b["classes"] for b in balls]))

    def predict_proba(self, X):
        X_arr = X.values if hasattr(X, "values") else np.asarray(X)
        nearest = np.argmin(cdist(X_arr, self.centers), axis=1)
        proba = np.zeros((len(X_arr), len(self.classes_)))
        for i, b_idx in enumerate(nearest):
            b = self.balls[b_idx]
            counts = np.clip(b["noisy_class_counts"], 0, None)
            total = counts.sum()
            p = counts / total if total > 0 else np.ones(len(counts)) / len(counts)
            for c, pc in zip(b["classes"], p):
                proba[i, np.where(self.classes_ == c)[0][0]] = pc
        return proba

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

# -----------------------------------------------------------------------------

main_results = []

for ds_name in NUMERIC_DATASETS:
    if ds_name not in datasets:
        continue
    X, y = datasets[ds_name]
    is_binary = y.nunique() == 2
    sens_col_idx = resolve_sens_col_idx(X, ds_name)
    sens_col_name = X.columns[sens_col_idx]        # <-- add this line

    # Public bounds from schema (Cell 2b); no privacy budget spent
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.3, random_state=seed, stratify=y
            )
        except ValueError:
            continue  # class too small to stratify at this split

        # Public-bounds scaling replaces private MinMax
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
        X_te_n = scale_with_public_bounds(X_te, lo, hi)

        X_tr_flat = make_flat_noised(X_tr_n, TOTAL_EPS, rng)
        # X_te is never part of the private release -- don't noise it. DP-GBC/DPLC/DP-SGD
        # all score against clean X_te_n; flat must match or the comparison is rigged.

        X_tr_flat_aia = X_tr_flat.copy()
        X_tr_flat_aia[sens_col_name] = X_tr_n[sens_col_name].values   # attacker's auxiliary label must be real, not noised-proxy

        # --- Baseline A: flat noise + LR ---
        util_a, model_a = evaluate_utility(X_tr_flat, y_tr, X_te_n, y_te)
        mia_a_lr = membership_inference_attack(model_a, X_tr_n, y_tr, X_te_n, y_te)
        mia_a_rf, _ = membership_inference_attack_rf(X_tr_flat, y_tr, X_te_n, y_te, seed)
        aia_a = run_aia(X_tr_flat_aia, y_tr, X_te_n, y_te, model_a, seed=seed, sens_col_idx=sens_col_idx) if is_binary else np.nan

        # --- Baseline B: flat noise + RF (naive "dp_forest") ---
        rf_b = RandomForestClassifier(n_estimators=100, max_depth=6,
                                       class_weight="balanced", random_state=seed)
        rf_b.fit(X_tr_flat, y_tr)
        util_b = {"balanced_accuracy": balanced_accuracy_score(y_te, rf_b.predict(X_te_n))}
        mia_b_rf, _ = membership_inference_attack_rf(X_tr_flat, y_tr, X_te_n, y_te, seed)

        # --- Condition C: DP-GBC (this paper's method, direct evaluation) ---
        max_depth = 3
        eps_count = eps_split = 0.1
        eps_center = 0.0      # public
        eps_radius = 0.0      # public
        eps_count_release = TOTAL_EPS - max_depth * (eps_count + eps_split)

        X_tr_c, X_te_c, y_tr_c, y_te_c, ledger, boundary_rate, balls = run_dp_gbc_pipeline(
            X, y, seed, lo, hi, max_depth=max_depth,
            eps_count_per_level=eps_count,
            eps_split_per_level=eps_split,
            eps_center_per_level=eps_center,
            eps_radius_per_level=eps_radius,
            eps_count_release=eps_count_release,
            all_classes=classes_public[ds_name]
        )

        # Assert total budget equals TOTAL_EPS exactly (bounds are public, no spend)
        assert abs(ledger.total_eps - TOTAL_EPS) < 1e-6

        preds_c_direct = dp_gbc_classify_predict(balls, X_te_c)

        # FIX 1b (corrected): restore the class-balanced oversampling Cell 5
        # documents as necessary -- Cell 7's simpler evaluate_utility() silently
        # dropped it, so the MLP was being fit unbalanced while LR (class_weight=
        # 'balanced') was not, making the LR-vs-MLP architecture check in Cell 9
        # not actually apples-to-apples as the comments claimed.
        from sklearn.neural_network import MLPClassifier
        from sklearn.utils import resample
        df_tr_c = X_tr_c.copy()
        y_tr_c_arr = y_tr_c.values if hasattr(y_tr_c, "values") else np.asarray(y_tr_c)
        df_tr_c["_y"] = y_tr_c_arr
        parts_c = [df_tr_c[df_tr_c["_y"] == c] for c in np.unique(y_tr_c_arr)]
        max_n_c = max(len(p) for p in parts_c)
        balanced_tr_c = pd.concat(
            [resample(p, replace=True, n_samples=max_n_c, random_state=seed) for p in parts_c]
        )
        mlp_c = MLPClassifier(hidden_layer_sizes=(16,), max_iter=500, random_state=seed)
        mlp_c.fit(balanced_tr_c.drop(columns="_y"), balanced_tr_c["_y"])
        util_c = {
            "balanced_accuracy": balanced_accuracy_score(y_te_c, preds_c_direct),
            "mlp_balanced_accuracy": balanced_accuracy_score(y_te_c, mlp_c.predict(X_te_c))
        }

        # ---- FIX 6: MIA attacks on DP-GBC now use REAL training/test data ----
        model_c_wrapper = BallModelWrapper(balls)
        mia_c_lr = membership_inference_attack(model_c_wrapper, X_tr_n, y_tr, X_te_n, y_te)   # correct as-is, queries the actual model
        mia_c_rf, _ = membership_inference_attack_rf_synth(X_tr_c, y_tr_c, X_tr_n, y_tr, X_te_n, y_te, seed)  # FIX: real members/non-members, not synthetic rows
        aia_c = run_aia(X_tr_c, y_tr_c, X_te_c, y_te_c, model_c_wrapper,
                        seed=seed, sens_col_idx=sens_col_idx) if is_binary else np.nan
        # ------------------------------------------------------------------------

        # --- Condition D: same release as Condition C, evaluated with a downstream classifier ---
        # (Confirmed identical release: same seed, same budget split, same call sequence as run_dp_gbc_pipeline.
        #  No second pipeline call needed -- reuse X_tr_c/y_tr_c/X_te_c/y_te_c directly.)
        util_d, model_d = evaluate_utility(X_tr_c, y_tr_c, X_te_c, y_te_c)
        mia_d_rf, _ = membership_inference_attack_rf_synth(X_tr_c, y_tr_c, X_tr_n, y_tr, X_te_n, y_te, seed)
        mia_d_lr = membership_inference_attack(model_d, X_tr_n, y_tr, X_te_n, y_te)
        mean_radius = np.mean([b["radius"] for b in balls])

        # positive control (once per dataset/seed, RAW undefended data)
        mia_control = positive_control_mia(X_tr_n.values, y_tr, X_te_n.values, y_te, seed)

        # NEW diagnostic: class collapse (no forced injection)
        class_collapse = int(pd.Series(y_tr_c).nunique() < len(np.unique(y)))

        main_results.append({
            "Dataset": ds_name, "Seed": seed, "n_classes": y.nunique(), "total_eps": TOTAL_EPS,
            "flat_LR_bal_acc": util_a["balanced_accuracy"], "flat_LR_auprc": util_a["auprc"],
            "flat_LR_MIA": mia_a_lr,
            "flat_LR_MIA_rf": mia_a_rf, "flat_LR_AIA": aia_a,
            "flat_RF_bal_acc": util_b["balanced_accuracy"], "flat_RF_MIA_rf": mia_b_rf,
            "dpgbc_bal_acc": util_c["balanced_accuracy"], "dpgbc_mlp_bal_acc": util_c["mlp_balanced_accuracy"],
            "dpgbc_MIA": mia_c_lr,
            "dpgbc_MIA_rf": mia_c_rf, "dpgbc_AIA": aia_c,
            "boundary_rate": boundary_rate, "MIA_positive_control": mia_control,
            "dplc_bal_acc": util_d["balanced_accuracy"], "dplc_auprc": util_d["auprc"],
            "dplc_MIA": mia_d_lr,
            "dplc_MIA_rf": mia_d_rf,
            "dplc_mean_radius": mean_radius,
            "class_collapse": class_collapse,
        })
        print(f"done: {ds_name} / seed {seed}")

df_main = pd.DataFrame(main_results)
df_main.to_csv("dpgbc_main_results.csv", index=False)
print(df_main.to_string(index=False))

In [ ]:
# Cell 8 (REPLACE): epsilon sweep across numeric-only datasets.
# Fixed: public bounds, fixed budget allocation, and updated pipeline signatures.
# Also: guards against empty synthetic outputs at very low TOTAL_EPS.
# Fix 2: loop variable renamed to eps_val so TOTAL_EPS is never overwritten.

TOTAL_EPS_GRID = [0.5, 1.0, 3.0, 8.0]
SWEEP_MAX_TRAIN_ROWS = 3000  # subsample large training sets for sweep speed only;
                              # main comparison (Cell 7) uses full data

NUMERIC_DATASETS = [
    "BreastCancer",
    "PimaDiabetes",
    "BloodTransfusion",
    "MAGIC04",
    "Banknote",
    "Ionosphere",
    "Sonar",
    "Wine",
    "WisconsinOriginal",
    "Waveform",
    "Vehicle",
    "Segment",
    "Spambase",
    "WineQuality",
]

# Precompute public bounds once per dataset (no privacy budget spent)
bounds_cache = {}
for ds_name in NUMERIC_DATASETS:
    if ds_name in datasets:
        X, y = datasets[ds_name]
        bounds_cache[ds_name] = public_bounds_for(ds_name, X)


def allocate_granulation_budget(total_eps, max_depth=3, min_count_release=0.05):
    """
    Allocate eps_count, eps_split, eps_count_release so that
    max_depth*(eps_count+eps_split) + eps_count_release == total_eps.
    eps_center and eps_radius are now 0.0 (public post-processing).
    """
    base_count = 0.1
    base_split = 0.1
    base_granulation = max_depth * (base_count + base_split)   # 0.6

    desired_count_release = max(min_count_release, 0.15 * total_eps)
    available_for_granulation = max(0.0, total_eps - desired_count_release)

    if available_for_granulation >= base_granulation:
        eps_count = base_count
        eps_split = base_split
        eps_count_release = total_eps - base_granulation
    else:
        scale = available_for_granulation / base_granulation
        eps_count = max(base_count * scale, 1e-6)
        eps_split = max(base_split * scale, 1e-6)
        actual_granulation = max_depth * (eps_count + eps_split)
        eps_count_release = max(total_eps - actual_granulation, desired_count_release)

    return eps_count, eps_split, 0.0, 0.0, eps_count_release


sweep_results = []

for eps_val in TOTAL_EPS_GRID:          # <-- Fix 2: renamed loop variable
    for ds_name in NUMERIC_DATASETS:
        if ds_name not in datasets:
            continue
        X, y = datasets[ds_name]
        lo, hi = bounds_cache[ds_name]

        for seed in SEEDS:
            rng = np.random.default_rng(seed)
            try:
                X_tr, X_te, y_tr, y_te = train_test_split(
                    X, y, test_size=0.3, random_state=seed, stratify=y
                )
            except ValueError:
                continue
            if len(X_tr) > SWEEP_MAX_TRAIN_ROWS:
                X_tr = X_tr.iloc[:SWEEP_MAX_TRAIN_ROWS]
                y_tr = y_tr.iloc[:SWEEP_MAX_TRAIN_ROWS]

            # ---- NEW: freeze the (possibly truncated) split for all conditions ----
            fixed_split = (X_tr, X_te, y_tr, y_te)

            # Public-bounds scaling (no private MinMax)
            X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
            X_te_n = scale_with_public_bounds(X_te, lo, hi)

            # Flat (baseline) -- only train features are noised
            X_tr_flat = make_flat_noised(X_tr_n, eps_val, rng)   # <-- Fix 2
            util_a, model_a = evaluate_utility(
                X_tr_flat, y_tr, X_te_n, y_te
            )
            mia_a, _ = membership_inference_attack_rf(
                X_tr_flat, y_tr, X_te_n, y_te, seed
            )

            # ---------- DP-GBC ----------
            max_depth = 3
            (eps_count, eps_split, eps_center,
             eps_radius, eps_count_release) = allocate_granulation_budget(
                eps_val, max_depth=max_depth          # <-- Fix 2
            )

            X_tr_c, X_te_c, y_tr_c, y_te_c, ledger, _, balls_c = run_dp_gbc_pipeline(
                X, y, seed, lo, hi,
                max_depth=max_depth,
                eps_count_per_level=eps_count,
                eps_split_per_level=eps_split,
                eps_center_per_level=eps_center,
                eps_radius_per_level=eps_radius,
                eps_count_release=eps_count_release,
                all_classes=classes_public[ds_name],
                precomputed_split=fixed_split          # NEW
            )
            # Bounds are public, so only the pipeline budget must equal eps_val
            assert abs(ledger.total_eps - eps_val) < 1e-6   # <-- Fix 2

            # Guard against empty synthetic output at extremely low eps.
            if len(X_tr_c) == 0:
                sweep_results.append({
                    "TOTAL_EPS": eps_val,              # <-- Fix 2
                    "Dataset": ds_name,
                    "Seed": seed,
                    "util_gap": np.nan,
                    "mia_gap": np.nan,
                    "dplc_util_gap": np.nan,
                    "dplc_mia_gap": np.nan,
                })
                continue

            # Option C: direct ball classification (no synthetic-data model)
            preds_c = dp_gbc_classify_predict(balls_c, X_te_c)
            util_c = {
                "balanced_accuracy": balanced_accuracy_score(y_te_c, preds_c),
                "mlp_balanced_accuracy": np.nan
            }
            # RF MIA on DP-GBC's own synthetic release
            mia_c, _ = membership_inference_attack_rf_synth(X_tr_c, y_tr_c, X_tr_n, y_tr, X_te_c, y_te_c, seed)

            # ---- Fix 6: add class_collapse diagnostic ----
            class_collapse = int(pd.Series(y_tr_c).nunique() < len(np.unique(y)))

            sweep_results.append({
                "TOTAL_EPS": eps_val,                  # <-- Fix 2
                "Dataset": ds_name,
                "Seed": seed,
                "util_gap": util_c["balanced_accuracy"] - util_a["balanced_accuracy"],
                "mia_gap": abs(mia_c - 0.5) - abs(mia_a - 0.5),
                "class_collapse": class_collapse,      # <-- Fix 6
            })

            # ---------- DP-LocalClip (alias of DP-GBC) ----------
            (eps_cnt_d, eps_spl_d, eps_ctr_d,
             eps_rad_d, eps_noise_lc) = allocate_granulation_budget(
                eps_val, max_depth=max_depth          # <-- Fix 2
            )

            X_tr_lc, X_te_lc, y_tr_lc, y_te_lc, ledger_lc, _ = run_dp_localclip_pipeline(
                X, y, seed, lo, hi,
                max_depth=max_depth,
                eps_count_per_level=eps_cnt_d,
                eps_split_per_level=eps_spl_d,
                eps_radius_per_level=eps_rad_d,
                eps_center_per_level=eps_ctr_d,
                eps_noise=eps_noise_lc,
                all_classes=classes_public[ds_name],
                precomputed_split=fixed_split          # NEW
            )
            # Bounds are public, so only the pipeline budget must equal eps_val
            assert abs(ledger_lc.total_eps - eps_val) < 1e-6   # <-- Fix 2

            if len(X_tr_lc) == 0:
                sweep_results[-1]["dplc_util_gap"] = np.nan
                sweep_results[-1]["dplc_mia_gap"] = np.nan
                continue

            util_lc, _ = evaluate_utility(
                X_tr_lc, y_tr_lc, X_te_lc, y_te_lc
            )
            mia_lc, _ = membership_inference_attack_rf_synth(X_tr_lc, y_tr_lc, X_tr_n, y_tr, X_te_lc, y_te_lc, seed)

            sweep_results[-1]["dplc_util_gap"] = (
                util_lc["balanced_accuracy"] - util_a["balanced_accuracy"]
            )
            sweep_results[-1]["dplc_mia_gap"] = abs(mia_lc - 0.5) - abs(mia_a - 0.5)

df_sweep = pd.DataFrame(sweep_results)
df_sweep.to_csv("dpgbc_sweep_results.csv", index=False)

# Summary for DP-GBC (no cert_frac anymore)
summary = df_sweep.groupby(["TOTAL_EPS"]).agg(
    mean_util_gap=("util_gap", "mean"),
    mean_mia_gap=("mia_gap", "mean"),
    mean_class_collapse=("class_collapse", "mean")    # optional summary
).reset_index()
wins = df_sweep.groupby(["TOTAL_EPS"]).apply(
    lambda g: ((g["util_gap"] > 0) & (g["mia_gap"] < 0)).mean()
).reset_index(name="dpgbc_win_frac")
summary = summary.merge(wins, on=["TOTAL_EPS"])

# Summary for DP-LocalClip
dplc_summary = df_sweep.groupby(["TOTAL_EPS"]).agg(
    dplc_mean_util_gap=("dplc_util_gap", "mean"),
    dplc_mean_mia_gap=("dplc_mia_gap", "mean")
).reset_index()
dplc_wins = df_sweep.groupby(["TOTAL_EPS"]).apply(
    lambda g: ((g["dplc_util_gap"] > 0) & (g["dplc_mia_gap"] < 0)).mean()
).reset_index(name="dplc_win_frac")

summary = summary.merge(dplc_summary, on=["TOTAL_EPS"])
summary = summary.merge(dplc_wins, on=["TOTAL_EPS"])

print(summary.to_string(index=False))

In [ ]:
# Cell 9 (FIXED): Friedman + Nemenyi + Holm-Bonferroni-corrected Wilcoxon,
# computed ONLY on the matched-epsilon df_main table from Cell 7 -- no
# epsilon-mixing bug this time, since every row already has total_eps==3.0.
# Statistical tests now use DATASET-level means (n_blocks = number of datasets).

# ---- Fix 3: Use ALL primary datasets for utility statistics ----
# Attack strength must never gate which datasets count toward the utility comparison.
valid_datasets = list(NUMERIC_DATASETS)   # NUMERIC_DATASETS is defined in Cell 7 (or earlier)
df_valid = df_main[df_main["Dataset"].isin(valid_datasets)].copy()

# Positive-control strength is a caveat on MIA results only -- it does not
# remove any dataset from the utility analysis.
pc_means = df_main.groupby("Dataset")["MIA_positive_control"].mean()
weak_positive_control = pc_means[pc_means <= 0.6].index.tolist()
print(f"All {len(valid_datasets)} primary datasets included in utility statistics.")
print(f"Datasets with a weak MIA positive control (<=0.6) -- report MIA "
      f"numbers for these with an explicit caveat: {weak_positive_control}")

from scipy.stats import friedmanchisquare, rankdata, wilcoxon, studentized_range
import statsmodels.stats.multitest as smm

def cliffs_delta(a, b):
    """Cliff's delta / matched-pairs dominance for paired data."""
    a, b = np.asarray(a), np.asarray(b)
    n = len(a)
    gt = np.sum(a > b)
    lt = np.sum(a < b)
    return (gt - lt) / n

methods = ["flat_LR_bal_acc", "flat_RF_bal_acc", "dpgbc_bal_acc", "dplc_bal_acc"]

# ---- Dataset-level aggregation (n_blocks = number of datasets) ----
pivot = df_valid.groupby("Dataset")[methods].mean().dropna()
n_blocks, k = len(pivot), len(methods)

# ---- Pre‑compute MIA pivot for DP-LocalClip (only applicable mechanism) ----
pivot_dplc = df_valid.groupby("Dataset")[["flat_LR_MIA_rf","dplc_MIA_rf"]].mean().dropna()

# ===== MIA ADVANTAGE FIX =====
pivot_dplc["flat_LR_MIA_rf_adv"] = (pivot_dplc["flat_LR_MIA_rf"] - 0.5).abs()
pivot_dplc["dplc_MIA_rf_adv"] = (pivot_dplc["dplc_MIA_rf"] - 0.5).abs()
# =============================

# ---- Friedman + average ranks ----
stat, p = friedmanchisquare(*[pivot[m] for m in methods])
print(f"Friedman (utility): chi2={stat:.3f}, p={p:.6f}")

ranks = np.apply_along_axis(lambda x: rankdata(-x), 1, pivot[methods].values)
avg_ranks = ranks.mean(axis=0)
print(pd.DataFrame({"Method": methods, "Avg Rank": avg_ranks}).sort_values("Avg Rank").to_string(index=False))

# ---- Nemenyi CD ----
se = np.sqrt(k * (k + 1) / (6 * n_blocks))
q = studentized_range.ppf(0.95, k, np.inf) / np.sqrt(2)
print(f"\nCD = {q*se:.4f}  (n_blocks={n_blocks})")

# ---- Pairwise Wilcoxon + Holm (utility) ----
pvals, labels = [], []
from itertools import combinations
for m1, m2 in combinations(methods, 2):
    stat, p = wilcoxon(pivot[m1], pivot[m2])
    pvals.append(p); labels.append(f"{m1} vs {m2}")
reject, p_holm, _, _ = smm.multipletests(pvals, method="holm")
for lab, p_raw, p_h, r in zip(labels, pvals, p_holm, reject):
    print(f"{lab:<35} raw p={p_raw:.5f}  Holm p={p_h:.5f}  {'SIGNIF' if r else 'not signif.'}")

# ---- Secondary check: architecture comparison on the SAME synthetic release ----
# Condition C (dpgbc_bal_acc) is a direct ball classifier.
# Condition D (dplc_bal_acc) is an LR trained on the SAME synthetic release as Condition C.
# dpgbc_mlp_bal_acc is an MLP trained on that same release.
# So this is a real LR-head vs MLP-head architecture check, using balanced accuracy.
piv_arch = df_valid.groupby("Dataset")[["dplc_bal_acc", "dpgbc_mlp_bal_acc"]].mean().dropna()

if len(piv_arch) == 0:
    print("\nArchitecture check (synthetic release): skipped (all values NaN)")
else:
    stat_a, p_a = wilcoxon(piv_arch["dplc_bal_acc"], piv_arch["dpgbc_mlp_bal_acc"])
    print(f"\nArchitecture check (synthetic release): LR-head vs MLP-head, Wilcoxon p={p_a:.5f}, "
          f"median diff = {np.median(piv_arch['dpgbc_mlp_bal_acc'] - piv_arch['dplc_bal_acc']):.4f}")

# ---- Effect size for the key utility comparison ----
d_util = cliffs_delta(pivot["dplc_bal_acc"].values, pivot["flat_LR_bal_acc"].values)
print(f"Cliff's delta (dplc vs flat utility): {d_util:.3f}  (|d|>0.33=medium, >0.47=large)")

# ===================== MIA block =====================
# NOTE: dplc here refers to the same release as DP-GBC (synthetic + LR).
# We use membership ADVANTAGE |AUC - 0.5| (Yeom et al. 2018):
# AUC far below 0.5 is also leakage because the attacker can simply flip predictions.
# cliffs_delta(dplc_MIA_rf_adv, flat_LR_MIA_rf_adv) > 0 means dplc leaks MORE.

# ---- Full set (all datasets) ----
print("\n===== MIA Results (all datasets) =====")
d_mia_full = cliffs_delta(
    pivot_dplc["dplc_MIA_rf_adv"].values,
    pivot_dplc["flat_LR_MIA_rf_adv"].values
)
print(f"Cliff's delta (dplc vs flat MIA advantage, full set): {d_mia_full:.3f}  "
      f"(positive = dplc has higher membership advantage / more leakage; |d|>0.33=medium, >0.47=large)")

stat_d_full, p_d_full = wilcoxon(
    pivot_dplc["flat_LR_MIA_rf_adv"],
    pivot_dplc["dplc_MIA_rf_adv"]
)
print(f"MIA_rf advantage: flat vs dplc (full set), Wilcoxon p={p_d_full:.5f}, "
      f"median diff = {np.median(pivot_dplc['dplc_MIA_rf_adv'] - pivot_dplc['flat_LR_MIA_rf_adv']):.4f} "
      f"(positive = dplc higher membership advantage)")

# Per-dataset MIA gap (positive = flat leaks more, dplc better; negative = dplc leaks more)
per_ds_mia_full = df_valid.groupby("Dataset").apply(
    lambda g: ((g["flat_LR_MIA_rf"] - 0.5).abs() - (g["dplc_MIA_rf"] - 0.5).abs()).mean()
).sort_values(ascending=False)
print("MIA advantage reduction per dataset (positive = dplc wins):")
print(per_ds_mia_full.to_string())

# ---- Excluding weak positive-control datasets ----
if weak_positive_control:
    # Filter pivot_dplc to only datasets with strong positive control (>0.6)
    strong_datasets = [d for d in pivot_dplc.index if d not in weak_positive_control]
    pivot_dplc_strong = pivot_dplc.loc[strong_datasets]
    if len(pivot_dplc_strong) >= 2:
        print(f"\n===== MIA Results (excluding {len(weak_positive_control)} weak-positive-control datasets) =====")
        d_mia_strong = cliffs_delta(
            pivot_dplc_strong["dplc_MIA_rf_adv"].values,
            pivot_dplc_strong["flat_LR_MIA_rf_adv"].values
        )
        print(f"Cliff's delta (dplc vs flat MIA advantage, strong positive-control only): {d_mia_strong:.3f}")

        stat_d_strong, p_d_strong = wilcoxon(
            pivot_dplc_strong["flat_LR_MIA_rf_adv"],
            pivot_dplc_strong["dplc_MIA_rf_adv"]
        )
        print(f"MIA_rf advantage: flat vs dplc (strong positive-control only), Wilcoxon p={p_d_strong:.5f}, "
              f"median diff = {np.median(pivot_dplc_strong['dplc_MIA_rf_adv'] - pivot_dplc_strong['flat_LR_MIA_rf_adv']):.4f}")

        # Per-dataset MIA gap for strong positive-control datasets
        df_valid_strong = df_valid[~df_valid["Dataset"].isin(weak_positive_control)]
        per_ds_mia_strong = df_valid_strong.groupby("Dataset").apply(
            lambda g: ((g["flat_LR_MIA_rf"] - 0.5).abs() - (g["dplc_MIA_rf"] - 0.5).abs()).mean()
        ).sort_values(ascending=False)
        print("MIA advantage reduction per dataset (strong positive-control only, positive = dplc wins):")
        print(per_ds_mia_strong.to_string())
    else:
        print(f"\nNot enough datasets with strong positive control for separate MIA analysis (only {len(pivot_dplc_strong)}).")
else:
    print("\nAll datasets have strong positive control; no separate subset analysis needed.")
# =====================================================================

In [ ]:
# Cell: Fidelity metrics for DP-GBC synthetic release
# Recomputes synthetic data for each dataset/seed and evaluates fidelity vs. real training data.
# Save results to CSV.

from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
import itertools

# -------------------------------
# Fidelity metric functions
# -------------------------------
def marginal_ks_error(real, synth):
    """Average KS statistic over features."""
    ks_vals = []
    for col in real.columns:
        ks_stat, _ = ks_2samp(real[col], synth[col])
        ks_vals.append(ks_stat)
    return float(np.mean(ks_vals))

def marginal_wasserstein_error(real, synth):
    """Average 1D Wasserstein distance over features."""
    w_vals = []
    for col in real.columns:
        w_dist = wasserstein_distance(real[col], synth[col])
        w_vals.append(w_dist)
    return float(np.mean(w_vals))

def pairwise_correlation_mae(real, synth):
    """Mean absolute error between correlation matrices."""
    corr_real = real.corr().values
    corr_synth = synth.corr().values
    # Take lower triangle (excluding diagonal) for MAE
    iu = np.triu_indices_from(corr_real, k=1)
    mae = np.mean(np.abs(corr_real[iu] - corr_synth[iu]))
    return float(mae)

def tstr_trtr_accuracy(X_synth, y_synth, X_real_train, y_real_train, X_real_test, y_real_test, seed):
    """
    Train on synthetic, test on real (TSTR) vs Train on real, test on real (TRTR).
    Returns both accuracies.
    """
    # TSTR
    clf_synth = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=seed)
    clf_synth.fit(X_synth, y_synth)
    tstr_acc = balanced_accuracy_score(y_real_test, clf_synth.predict(X_real_test))

    # TRTR
    clf_real = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=seed)
    clf_real.fit(X_real_train, y_real_train)
    trtr_acc = balanced_accuracy_score(y_real_test, clf_real.predict(X_real_test))
    return tstr_acc, trtr_acc

# -------------------------------
# Main fidelity loop
# -------------------------------
# Use same configuration as Cell 7
max_depth = 3
eps_count = eps_split = 0.1
eps_center = eps_radius = 0.0
eps_count_release = TOTAL_EPS - max_depth * (eps_count + eps_split)

fidelity_results = []

for ds_name in NUMERIC_DATASETS:
    if ds_name not in datasets:
        continue
    X, y = datasets[ds_name]
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.3, random_state=seed, stratify=y
            )
        except ValueError:
            continue

        # Scale using public bounds
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
        X_te_n = scale_with_public_bounds(X_te, lo, hi)

        # Run DP-GBC pipeline (same as Cell 7)
        X_tr_c, X_te_c, y_tr_c, y_te_c, ledger, boundary_rate, balls = run_dp_gbc_pipeline(
            X, y, seed, lo, hi,
            max_depth=max_depth,
            eps_count_per_level=eps_count,
            eps_split_per_level=eps_split,
            eps_center_per_level=eps_center,
            eps_radius_per_level=eps_radius,
            eps_count_release=eps_count_release,
            all_classes=classes_public[ds_name]
        )

        # Skip if synthetic set empty
        if len(X_tr_c) == 0:
            continue

        # Compute metrics (real vs synthetic training sets, both scaled)
        ks_err = marginal_ks_error(X_tr_n, X_tr_c)
        wass_err = marginal_wasserstein_error(X_tr_n, X_tr_c)
        corr_mae = pairwise_correlation_mae(X_tr_n, X_tr_c)

        # TSTR/TRTR (using real test set)
        tstr_acc, trtr_acc = tstr_trtr_accuracy(
            X_tr_c, y_tr_c, X_tr_n, y_tr, X_te_n, y_te, seed
        )

        fidelity_results.append({
            "Dataset": ds_name,
            "Seed": seed,
            "marginal_ks": ks_err,
            "marginal_wasserstein": wass_err,
            "correlation_mae": corr_mae,
            "tstr_acc": tstr_acc,
            "trtr_acc": trtr_acc,
            "tstr_minus_trtr": tstr_acc - trtr_acc
        })
        print(f"done fidelity: {ds_name} / seed {seed}")

df_fidelity = pd.DataFrame(fidelity_results)
df_fidelity.to_csv("dpgbc_fidelity_metrics.csv", index=False)

# Summary statistics
print("\nFidelity metrics summary (mean across seeds per dataset):")
fidelity_summary = df_fidelity.groupby("Dataset")[
    ["marginal_ks", "marginal_wasserstein", "correlation_mae", "tstr_acc", "trtr_acc", "tstr_minus_trtr"]
].mean()
print(fidelity_summary.to_string())

In [ ]:
# Cell 9b (NEW): Sampling-geometry ablation — box-uniform vs. box-Gaussian.
# Both modes spend IDENTICAL privacy budget: sampling shape is a deterministic,
# public post-processing function of the already-released box bounds, so this
# is a fair, zero-extra-cost comparison. Reuses run_dp_gbc_pipeline (Cell 4)
# unchanged via the new sampling_mode parameter.

SAMPLING_ABLATION_MODES = ["uniform", "gaussian"]
ablation_sampling_results = []

_max_depth = 3
_eps_count = _eps_split = 0.1
_eps_center = _eps_radius = 0.0
_eps_count_release = TOTAL_EPS - _max_depth * (_eps_count + _eps_split)

for ds_name in NUMERIC_DATASETS:
    if ds_name not in datasets:
        continue
    X, y = datasets[ds_name]
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.3, random_state=seed, stratify=y
            )
        except ValueError:
            continue

        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
        X_te_n = scale_with_public_bounds(X_te, lo, hi)
        fixed_split = (X_tr, X_te, y_tr, y_te)  # SAME split for both modes

        for mode in SAMPLING_ABLATION_MODES:
            X_tr_c, X_te_c, y_tr_c, y_te_c, ledger, boundary_rate, balls = run_dp_gbc_pipeline(
                X, y, seed, lo, hi,
                max_depth=_max_depth,
                eps_count_per_level=_eps_count,
                eps_split_per_level=_eps_split,
                eps_center_per_level=_eps_center,
                eps_radius_per_level=_eps_radius,
                eps_count_release=_eps_count_release,
                all_classes=classes_public[ds_name],
                precomputed_split=fixed_split,
                sampling_mode=mode,
            )
            assert abs(ledger.total_eps - TOTAL_EPS) < 1e-6

            if len(X_tr_c) == 0:
                continue

            ks_err = marginal_ks_error(X_tr_n, X_tr_c)
            wass_err = marginal_wasserstein_error(X_tr_n, X_tr_c)
            corr_mae = pairwise_correlation_mae(X_tr_n, X_tr_c)
            tstr_acc, trtr_acc = tstr_trtr_accuracy(
                X_tr_c, y_tr_c, X_tr_n, y_tr, X_te_n, y_te, seed
            )
            util, model = evaluate_utility(X_tr_c, y_tr_c, X_te_c, y_te_c)
            mia_rf, _ = membership_inference_attack_rf_synth(
                X_tr_c, y_tr_c, X_tr_n, y_tr, X_te_n, y_te, seed
            )

            ablation_sampling_results.append({
                "Dataset": ds_name, "Seed": seed, "sampling_mode": mode,
                "marginal_ks": ks_err, "marginal_wasserstein": wass_err,
                "correlation_mae": corr_mae,
                "tstr_acc": tstr_acc, "trtr_acc": trtr_acc,
                "tstr_minus_trtr": tstr_acc - trtr_acc,
                "bal_acc": util["balanced_accuracy"], "mia_rf": mia_rf,
            })
        print(f"done sampling-ablation: {ds_name} / seed {seed}")

df_sampling_ablation = pd.DataFrame(ablation_sampling_results)
df_sampling_ablation.to_csv("dpgbc_sampling_ablation.csv", index=False)

summary_sampling = df_sampling_ablation.groupby(["Dataset", "sampling_mode"])[
    ["marginal_ks", "marginal_wasserstein", "correlation_mae",
     "tstr_minus_trtr", "bal_acc", "mia_rf"]
].mean().round(3)
print("\n=== Box-uniform vs. box-Gaussian: fidelity + utility + MIA ===")
print(summary_sampling.to_string())

from scipy.stats import wilcoxon as _wilcoxon
piv = df_sampling_ablation.groupby(["Dataset", "sampling_mode"])[
    ["marginal_ks", "correlation_mae", "bal_acc", "mia_rf"]
].mean().unstack("sampling_mode")
print("\n=== Wilcoxon, gaussian vs. uniform (dataset-level means, n=14) ===")
for metric in ["marginal_ks", "correlation_mae", "bal_acc", "mia_rf"]:
    u = piv[(metric, "uniform")].dropna()
    g = piv[(metric, "gaussian")].dropna()
    common = u.index.intersection(g.index)
    if len(common) >= 2:
        stat, p = _wilcoxon(u.loc[common], g.loc[common])
        print(f"{metric:<16} median(gaussian-uniform) = "
              f"{(g.loc[common]-u.loc[common]).median():+.4f}   p={p:.4f}")

In [ ]:
# Cell 10 (NEW): reviewer-facing checklist, tied to citable reporting
# arXiv 2508.15141 (Bao & Bindschaedler, ACSAC 2024) reliability/generalizability axes;
# NIST DP guidance on
# documenting accountant + parameters). Run this and paste the printout
# directly into your methodology/limitations section.

print("=== Reviewer checklist ===\n")

print(f"1) Datasets used: {len(datasets)} (target: 13)")
print(f"   -> generalizability axis: multiple domains, sizes, imbalance levels\n")

print(f"2) Privacy regimes tested: {sorted(set(sweep_results and df_sweep['TOTAL_EPS'].unique()))}")
print(f"   -> generalizability axis: not a single cherry-picked epsilon\n")

print(f"3) Positive control (attack sanity check), per dataset:")
pc = df_main.groupby("Dataset")["MIA_positive_control"].mean().sort_values()
print(pc.to_string())
print(f"   mean = {pc.mean():.3f}  (should be > ~0.55; if any dataset is near 0.5,")
print(f"   flag that dataset's MIA results as uninterpretable, don't report them as 'no leakage')\n")

# FIX: read the actual eps from the data, not the stale global
expected_eps = df_main["total_eps"].iloc[0]
print(f"4) Budget honesty: all rows should show total_eps == {expected_eps:.1f}")
print(f"   mismatches: {(df_main['total_eps'] != expected_eps).sum()} (should be 0)\n")

print(f"5) Utility collapse check (balanced accuracy):")
print(df_valid.groupby("Dataset")[
    ["flat_LR_bal_acc","dpgbc_bal_acc","dplc_bal_acc"]
].mean().to_string())

print(f"\n6) AIA lift -- did dplc reduce attribute leakage vs flat?")
aia_binary = df_valid.dropna(subset=["flat_LR_AIA","dpgbc_AIA"])
print(aia_binary.groupby("Dataset")[["flat_LR_AIA","dpgbc_AIA"]].mean().to_string())

In [ ]:
# Cell 11 (REPLACE): Tradeoff characterization — the paper's actual contribution
# Question: which datasets benefit from dplc's MIA protection despite the utility cost?

# First tradeoff DataFrame (overwritten below; kept for consistency)
tradeoff = df_valid.groupby("Dataset").agg(
    util_cost=("dplc_bal_acc", lambda g: (g - df_valid.loc[g.index, "flat_LR_bal_acc"]).mean()),
    mia_gain=("dplc_MIA_rf", lambda g: (
        (df_valid.loc[g.index, "flat_LR_MIA_rf"] - 0.5).abs() - (g - 0.5).abs()
    ).mean()),
    flat_util=("flat_LR_bal_acc", "mean"),
    flat_mia=("flat_LR_MIA_rf", "mean"),
    dplc_util=("dplc_bal_acc", "mean"),
    dplc_mia=("dplc_MIA_rf", "mean"),
    silhouette=("Dataset", lambda g: pd.Series(clusterability).reindex(g).values[0])
).reset_index()

# Note: util_cost is negative (dplc loses utility);
# mia_gain is positive when dplc reduces membership ADVANTAGE |AUC-0.5|.
# AUC far below 0.5 is also leakage because the attacker can flip predictions.
tradeoff = pd.DataFrame({
    "Dataset": df_valid.groupby("Dataset")["flat_LR_bal_acc"].mean().index,
    "util_cost": df_valid.groupby("Dataset").apply(
        lambda g: (g["dplc_bal_acc"] - g["flat_LR_bal_acc"]).mean()
    ),
    "mia_gain": df_valid.groupby("Dataset").apply(
        lambda g: ((g["flat_LR_MIA_rf"] - 0.5).abs() - (g["dplc_MIA_rf"] - 0.5).abs()).mean()
    ),
    "flat_mia": df_valid.groupby("Dataset")["flat_LR_MIA_rf"].mean(),
    "flat_util": df_valid.groupby("Dataset")["flat_LR_bal_acc"].mean(),
}).reset_index(drop=True)

tradeoff["tradeoff_ratio"] = tradeoff["mia_gain"] / tradeoff["util_cost"].abs().clip(lower=1e-6)
# ratio > 0 means mia_gain outweighs utility cost per unit
# ratio > 1 means you gain more MIA protection than you lose utility (favorable tradeoff)

tradeoff = tradeoff.sort_values("mia_gain", ascending=False)
print("=== Tradeoff: MIA advantage gain vs utility cost per dataset ===")
print(tradeoff[["Dataset","util_cost","mia_gain","tradeoff_ratio","flat_mia"]].to_string(index=False))

print("\nDatasets where MIA advantage gain > utility cost (ratio > 1):")
favorable = tradeoff[tradeoff["tradeoff_ratio"] > 1]
print(favorable[["Dataset","util_cost","mia_gain","tradeoff_ratio"]].to_string(index=False))

print(f"\nDatasets where flat_mia is HIGH (attack was strongest, dplc most useful):")
print(tradeoff[tradeoff["flat_mia"] > 0.65][
    ["Dataset","flat_mia","mia_gain","util_cost"]
].sort_values("flat_mia", ascending=False).to_string(index=False))

# ----------------------------------------------------------------------
# ADDITIONAL ANALYSIS: Per‑dataset Pareto dominance
# Measures how often (across seeds) dplc beats flat on BOTH axes:
#   - utility: dplc_bal_acc >= flat_LR_bal_acc
#   - privacy: |dplc_MIA_rf - 0.5| <= |flat_LR_MIA_rf - 0.5|
# ----------------------------------------------------------------------
dominance = df_valid.groupby("Dataset").apply(
    lambda g: (
        (g["dplc_bal_acc"] >= g["flat_LR_bal_acc"]) &
        ((g["dplc_MIA_rf"] - 0.5).abs() <= (g["flat_LR_MIA_rf"] - 0.5).abs())
    ).mean()
).rename("dplc_dominance_frac")
print("\n=== Per-dataset Pareto dominance (dplc beats flat on BOTH axes) ===")
print(dominance.sort_values(ascending=False).to_string())

In [ ]:
# Cell 12 (NEW): Fix the silently-dropped silhouette column from Cell 11 and
# actually run the correlation analysis you intended.
from scipy.stats import spearmanr

tradeoff["silhouette"] = tradeoff["Dataset"].map(clusterability)
print("=== Tradeoff table with silhouette (corrected) ===")
print(tradeoff[["Dataset","util_cost","mia_gain","tradeoff_ratio","silhouette"]]
      .sort_values("silhouette", ascending=False).to_string(index=False))

r_util, p_util = spearmanr(tradeoff["silhouette"], tradeoff["util_cost"])
r_mia, p_mia = spearmanr(tradeoff["silhouette"], tradeoff["mia_gain"])
print(f"\nSpearman: silhouette vs util_cost: r={r_util:.3f}, p={p_util:.4f}")
print(f"Spearman: silhouette vs mia_gain:  r={r_mia:.3f}, p={p_mia:.4f}")
print("(n=12 datasets — descriptive/exploratory only, report honestly either way)")

In [ ]:
# Cell 13 (NEW): Non-private oracle baseline — the "privacy tax" row every DP
# utility paper needs. Uses IDENTICAL splits (same seed/stratify) as Cell 7,
# so it's directly comparable, not a separate experiment.
TOTAL_EPS = 3.0  # RESET: Cell 8's sweep loop leaves this at its last grid value (8.0)
                 # after finishing. Every cell below this point that references the
                 # bare TOTAL_EPS global was silently using eps=8, not eps=3, until now.

oracle_results = []
for ds_name, (X, y) in datasets.items():
    for seed in SEEDS:
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
        except ValueError:
            continue
        scaler = MinMaxScaler()
        X_tr_n = pd.DataFrame(scaler.fit_transform(X_tr), columns=X.columns, index=X_tr.index)
        X_te_n = pd.DataFrame(scaler.transform(X_te), columns=X.columns, index=X_te.index)
        util_oracle, _ = evaluate_utility(X_tr_n, y_tr, X_te_n, y_te)
        oracle_results.append({"Dataset": ds_name, "Seed": seed,
                                "oracle_bal_acc": util_oracle["balanced_accuracy"]})

df_oracle = pd.DataFrame(oracle_results)

# Ensure df_main exists (it should be created in Cell 7)
if "df_main" not in globals():
    print("WARNING: df_main not found. Did you run Cell 7? Creating df_main from oracle results only.")
    df_main = df_oracle.copy()
else:
    # Merge oracle results into main results
    df_main = df_main.merge(df_oracle, on=["Dataset", "Seed"], how="left")
    print("After merge, df_main columns:", df_main.columns.tolist())
    # Double-check that the merge succeeded
    if "oracle_bal_acc" not in df_main.columns:
        raise RuntimeError("Merge failed: 'oracle_bal_acc' column missing. Check dataset name consistency.")

# Filter to valid datasets (if valid_datasets is defined, else use all)
if "valid_datasets" in globals():
    df_valid = df_main[df_main["Dataset"].isin(valid_datasets)].copy()
else:
    print("WARNING: valid_datasets not defined; using all datasets.")
    df_valid = df_main.copy()

# Ensure the required columns exist in df_valid before grouping
required_cols = ["oracle_bal_acc", "flat_LR_bal_acc", "dpgbc_bal_acc", "dplc_bal_acc"]
missing = [col for col in required_cols if col not in df_valid.columns]
if missing:
    raise KeyError(f"Missing columns in df_valid: {missing}")

# Compute privacy tax
privacy_tax = df_valid.groupby("Dataset")[required_cols].mean()
privacy_tax["tax_flat"]  = privacy_tax["oracle_bal_acc"] - privacy_tax["flat_LR_bal_acc"]
privacy_tax["tax_dpgbc"] = privacy_tax["oracle_bal_acc"] - privacy_tax["dpgbc_bal_acc"]
privacy_tax["tax_dplc"]  = privacy_tax["oracle_bal_acc"] - privacy_tax["dplc_bal_acc"]

print("=== Privacy tax (oracle - private method), lower = better ===")
print(privacy_tax[["oracle_bal_acc", "tax_flat", "tax_dpgbc", "tax_dplc"]].round(3).to_string())

In [ ]:
# Cell 14 (UPDATED): Membership advantage and loss-based MIA.
# Fixes:
#   - public schema bounds instead of private dp_release_feature_bounds
#   - updated run_dp_localclip_pipeline signature with lo, hi
#   - budget allocation kept exact at TOTAL_EPS
#   - uses fixed synthetic DP-LocalClip release
#   - robust loss_based_mia handles missing classes
#   - FIX 6: DPLC MIA now scored on REAL data (X_tr_n, y_tr, X_te_n, y_te)

from sklearn.metrics import roc_curve, roc_auc_score
from scipy.stats import wilcoxon

def membership_advantage(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores)
    return float(np.max(tpr - fpr))

def loss_based_mia(model, X_tr, y_tr, X_te, y_te):
    """Robust version: handles cases where y contains classes not in model.classes_."""
    try:
        proba_tr = model.predict_proba(X_tr)
        proba_te = model.predict_proba(X_te)
    except Exception:
        return 0.5, 0.0

    classes = model.classes_
    class_to_idx = {c: i for i, c in enumerate(classes)}

    def nll(proba, y):
        y = np.asarray(y)
        p = np.full(len(y), 1e-12)
        for i, val in enumerate(y):
            if val in class_to_idx:
                idx = class_to_idx[val]
                p[i] = proba[i, idx]
        p = np.clip(p, 1e-12, 1.0)
        return -np.log(p)

    loss_tr = nll(proba_tr, y_tr.values if hasattr(y_tr, "values") else y_tr)
    loss_te = nll(proba_te, y_te.values if hasattr(y_te, "values") else y_te)

    scores = np.concatenate([-loss_tr, -loss_te])
    labels = np.concatenate([np.ones(len(loss_tr)), np.zeros(len(loss_te))])

    try:
        auc = roc_auc_score(labels, scores)
    except Exception:
        auc = 0.5
    return auc, membership_advantage(labels, scores)


TOTAL_EPS = float(globals().get("TOTAL_EPS", 3.0))

# Precompute public bounds once per dataset (no privacy budget spent)
bounds_cache = {}
for ds_name, (X, y) in datasets.items():
    bounds_cache[ds_name] = public_bounds_for(ds_name, X)

adv_results = []

for ds_name, (X, y) in datasets.items():
    lo, hi = bounds_cache[ds_name]

    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.3, random_state=seed, stratify=y
            )
        except ValueError:
            continue

        # Public-bounds scaling, no private MinMax
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
        X_te_n = scale_with_public_bounds(X_te, lo, hi)

        # ---------- Flat (baseline) ----------
        X_tr_flat = make_flat_noised(X_tr_n, TOTAL_EPS, rng)

        _, model_flat = evaluate_utility(X_tr_flat, y_tr, X_te_n, y_te)
        auc_lf, adv_f = loss_based_mia(
            model_flat, X_tr_n, y_tr, X_te_n, y_te
        )

        # ---------- DP-LocalClip (fixed synthetic release) ----------
        max_depth = 3
        eps_cnt = eps_spl = 0.1
        eps_ctr = 0.0      # public
        eps_rad = 0.0      # public
        eps_noise_d = TOTAL_EPS - max_depth * (eps_cnt + eps_spl)

        X_tr_d, X_te_d, y_tr_d, y_te_d, _, _ = run_dp_localclip_pipeline(
            X, y, seed, lo, hi,
            max_depth=max_depth,
            eps_count_per_level=eps_cnt,
            eps_split_per_level=eps_spl,
            eps_radius_per_level=eps_rad,
            eps_center_per_level=eps_ctr,
            eps_noise=eps_noise_d,
            all_classes=classes_public[ds_name]
        )

        if len(X_tr_d) == 0:
            adv_results.append({
                "Dataset": ds_name,
                "Seed": seed,
                "flat_loss_MIA_auc": auc_lf,
                "flat_MIA_advantage": adv_f,
                "dplc_loss_MIA_auc": np.nan,
                "dplc_MIA_advantage": np.nan,
            })
            continue

        try:
            _, model_dplc = evaluate_utility(X_tr_d, y_tr_d, X_te_d, y_te_d)
            auc_ld, adv_d = loss_based_mia(
                model_dplc, X_tr_n, y_tr, X_te_n, y_te
            )
        except Exception as e:
            print(f"Warning: DP-LocalClip evaluation failed for {ds_name} seed {seed}: {e}")
            auc_ld, adv_d = np.nan, np.nan

        adv_results.append({
            "Dataset": ds_name,
            "Seed": seed,
            "flat_loss_MIA_auc": auc_lf,
            "flat_MIA_advantage": adv_f,
            "dplc_loss_MIA_auc": auc_ld,
            "dplc_MIA_advantage": adv_d,
        })

df_adv = pd.DataFrame(adv_results)

# Use valid_datasets if defined, else all numeric-only datasets
if 'valid_datasets' in globals():
    valid_list = valid_datasets
else:
    valid_list = [
        "BreastCancer", "PimaDiabetes", "BloodTransfusion",
        "MAGIC04", "Banknote"
    ]

df_adv_valid = df_adv[df_adv["Dataset"].isin(valid_list)]

print(
    df_adv_valid.groupby("Dataset")[
        ["flat_loss_MIA_auc", "dplc_loss_MIA_auc",
         "flat_MIA_advantage", "dplc_MIA_advantage"]
    ].mean().round(3).to_string()
)

piv_adv = df_adv_valid.pivot_table(
    index=["Dataset", "Seed"],
    values=["flat_MIA_advantage", "dplc_MIA_advantage"],
    aggfunc="mean"
)

if len(piv_adv) > 0:
    valid_pairs = piv_adv.dropna()
    if len(valid_pairs) > 0:
        stat_adv, p_adv = wilcoxon(
            valid_pairs["flat_MIA_advantage"],
            valid_pairs["dplc_MIA_advantage"]
        )
        print(f"\nWilcoxon on membership advantage, flat vs dplc: p={p_adv:.5f}")
    else:
        print("\nNot enough valid pairs for Wilcoxon test.")
else:
    print("\nNo data for Wilcoxon test.")

In [ ]:
# Cell 15 (NEW): 95% CIs across seeds (t-distribution, honest at n=5) for the
# headline numbers — you have means everywhere but no error bars, and reviewers
# will ask whether a small gap is signal or seed noise.
from scipy.stats import t as t_dist

def mean_ci(x, conf=0.95):
    x = np.asarray(x, dtype=float); n = len(x)
    m, se = x.mean(), x.std(ddof=1) / np.sqrt(n)
    h = se * t_dist.ppf((1 + conf) / 2., n - 1) if n > 1 else np.nan
    return m, m - h, m + h

rows = []
for ds in valid_datasets:
    sub = df_valid[df_valid["Dataset"] == ds]
    for col, label in [("flat_LR_bal_acc","flat_util"), ("dplc_bal_acc","dplc_util"),
                        ("flat_LR_MIA_rf","flat_mia"), ("dplc_MIA_rf","dplc_mia")]:
        m, lo, hi = mean_ci(sub[col].values)
        rows.append({"Dataset": ds, "metric": label, "mean": m, "half_width": m - lo})

df_ci = pd.DataFrame(rows)
print(df_ci.pivot(index="Dataset", columns="metric", values="mean").round(3).to_string())
print("\n95% CI half-widths:")
print(df_ci.pivot(index="Dataset", columns="metric", values="half_width").round(3).to_string())

In [ ]:
# Cell 16 (UPDATED): Non-DP granular-ball ablation.
#
# Uses epsilon -> 1e6 to approximate a noiseless limit of the fixed
# count-then-sample DP-LocalClip mechanism. The released data are synthetic
# samples from granular balls, not direct record clipping.
#
# Fixes:
#   - public schema bounds instead of private dp_release_feature_bounds
#   - updated run_dp_localclip_pipeline signature with lo, hi
#   - precompute bounds once per dataset
#   - restrict to NUMERIC_DATASETS for consistency with main results

NONPRIVATE_EPS = 1e6
SEEDS = globals().get("SEEDS", [42, 123, 456, 789, 101])

# ---------------------------------------------------------------------
# NEW: define or import the list of fully numeric datasets used in Cell 7–15.
# If you already have a variable NUMERIC_DATASETS, skip this block.
if "NUMERIC_DATASETS" not in globals():
    NUMERIC_DATASETS = [
        "BreastCancer", "PimaDiabetes", "BloodTransfusion",
        "MAGIC04", "Banknote",
        # Add any newly loaded numeric datasets (Ionosphere, Sonar, Wine, etc.)
        "Ionosphere", "Sonar", "Wine", "WisconsinOriginal",
        "Waveform", "Vehicle", "Segment", "Spambase", "WineQuality"
    ]
    # Keep only those that actually loaded successfully
    NUMERIC_DATASETS = [d for d in NUMERIC_DATASETS if d in datasets]
# ---------------------------------------------------------------------

# Precompute public bounds once per dataset (no privacy budget spent)
bounds_cache = {}
for ds_name in NUMERIC_DATASETS:   # <-- changed from datasets.items()
    X, y = datasets[ds_name]
    bounds_cache[ds_name] = public_bounds_for(ds_name, X)

ablation_results = []

for ds_name in NUMERIC_DATASETS:   # <-- changed from datasets.items()
    X, y = datasets[ds_name]
    lo, hi = bounds_cache[ds_name]

    for seed in SEEDS:
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.3, random_state=seed, stratify=y
            )
        except ValueError:
            continue

        X_tr_d, X_te_d, y_tr_d, y_te_d, _, _ = run_dp_localclip_pipeline(
            X, y, seed, lo, hi,
            max_depth=3,
            eps_count_per_level=NONPRIVATE_EPS,
            eps_split_per_level=NONPRIVATE_EPS,
            eps_radius_per_level=NONPRIVATE_EPS,
            eps_center_per_level=NONPRIVATE_EPS,
            eps_noise=NONPRIVATE_EPS,
            all_classes=classes_public[ds_name]
        )

        if len(X_tr_d) == 0:
            ablation_results.append({
                "Dataset": ds_name,
                "Seed": seed,
                "nonpriv_localclip_bal_acc": np.nan,
            })
            continue

        util_np, _ = evaluate_utility(X_tr_d, y_tr_d, X_te_d, y_te_d)

        ablation_results.append({
            "Dataset": ds_name,
            "Seed": seed,
            "nonpriv_localclip_bal_acc": util_np["balanced_accuracy"],
        })

df_ablation = pd.DataFrame(ablation_results)

# Merge with oracle and DP-LocalClip results if available.
# If df_oracle or df_valid do not exist yet, this part will still run but
# the subsequent grouping will show NaN for missing columns.
if "df_oracle" in globals():
    df_ablation = df_ablation.merge(
        df_oracle, on=["Dataset", "Seed"], how="left"
    )

if "df_valid" in globals() and "dplc_bal_acc" in df_valid.columns:
    df_ablation = df_ablation.merge(
        df_valid[["Dataset", "Seed", "dplc_bal_acc"]],
        on=["Dataset", "Seed"],
        how="left"
    )

# Compute decomposition if required columns exist
required_cols = ["oracle_bal_acc", "nonpriv_localclip_bal_acc", "dplc_bal_acc"]
if all(col in df_ablation.columns for col in required_cols):
    comp_structure_noise = df_ablation.groupby("Dataset")[required_cols].mean()
    comp_structure_noise["structure_cost"] = (
        comp_structure_noise["oracle_bal_acc"] - comp_structure_noise["nonpriv_localclip_bal_acc"]
    )
    comp_structure_noise["noise_cost"] = (
        comp_structure_noise["nonpriv_localclip_bal_acc"] - comp_structure_noise["dplc_bal_acc"]
    )
    print("=== Decomposing dplc's utility cost: structure vs. noise ===")
    print(comp_structure_noise.round(3).to_string())
else:
    print("Missing required result columns; skipping cost decomposition.")
    print(df_ablation.head())

In [ ]:
# Cell 17 (UPDATED): Wall-clock cost.
# Fixes:
#   - public bounds instead of private DP-released bounds
#   - updated run_dp_gbc_pipeline and run_dp_localclip_pipeline signatures
#   - precompute public bounds once per dataset

import time

TOTAL_EPS = float(globals().get("TOTAL_EPS", 3.0))

# Precompute public bounds once per dataset (no privacy budget spent)
bounds_cache = {}
for ds_name, (X, y) in datasets.items():
    bounds_cache[ds_name] = public_bounds_for(ds_name, X)

timing_results = []

for ds_name, (X, y) in datasets.items():
    lo, hi = bounds_cache[ds_name]
    seed = SEEDS[0]

    try:
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.3, random_state=seed, stratify=y
        )
    except ValueError:
        continue

    # Public-bounds scaling, no private MinMax
    X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
    rng = np.random.default_rng(seed)

    t0 = time.perf_counter()
    _ = make_flat_noised(X_tr_n, TOTAL_EPS, rng)
    t_flat = time.perf_counter() - t0

    t0 = time.perf_counter()
    _ = run_dp_gbc_pipeline(
        X, y, seed, lo, hi,
        max_depth=3,
        eps_count_per_level=0.1,
        eps_split_per_level=0.1,
        eps_center_per_level=0.1,
        eps_radius_per_level=0.4,
        eps_count_release=0.9
    )
    t_gbc = time.perf_counter() - t0

    t0 = time.perf_counter()
    _ = run_dp_localclip_pipeline(
        X, y, seed, lo, hi,
        max_depth=3,
        eps_count_per_level=0.1,
        eps_split_per_level=0.1,
        eps_radius_per_level=0.5,
        eps_center_per_level=0.1,
        eps_noise=1.5
    )
    t_dplc = time.perf_counter() - t0

    timing_results.append({
        "Dataset": ds_name,
        "n": len(X),
        "d": X.shape[1],
        "t_flat_sec": t_flat,
        "t_dpgbc_sec": t_gbc,
        "t_dplc_sec": t_dplc,
    })

print(pd.DataFrame(timing_results).round(4).to_string(index=False))

In [ ]:
# Cell 18 (FIXED again): adds (a) a per-fit timeout via SIGALRM so a hung MST
# fit can't block the whole loop, and (b) collinear-column pruning before
# fitting -- BloodTransfusion's Monetary = 25*Frequency is a known near-exact
# linear dependency, and MST's private-pgm junction-tree inference can stall
# on the resulting near-deterministic discretized marginals. SIGALRM is
# Unix-only (fine for Colab/Linux; won't work on native Windows).

try:
    import snsynth
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "smartnoise-synth"])
    import snsynth
from snsynth import Synthesizer
import time, signal
from sklearn.metrics import roc_auc_score  # Added for FIX 6 MIA computation

DPCTGAN_MAX_ROWS = 1500
MST_TIMEOUT_SEC = 90
DPCTGAN_TIMEOUT_SEC = 60

class TimeoutException(Exception):
    pass

def _timeout_handler(signum, frame):
    raise TimeoutException()

def run_with_timeout(fn, timeout_sec):
    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(timeout_sec)
    try:
        return fn()
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

def drop_collinear(X_tr, X_te, threshold=0.98):
    corr = X_tr.corr().abs()
    cols = list(corr.columns)
    to_drop = set()
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            if cols[j] not in to_drop and corr.iloc[i, j] > threshold:
                to_drop.add(cols[j])
    if to_drop:
        return X_tr.drop(columns=list(to_drop)), X_te.drop(columns=list(to_drop)), to_drop
    return X_tr, X_te, to_drop

def run_synth_baseline(X_tr, y_tr, X_te, y_te, eps, synth_name, seed):
    X_tr, X_te, dropped = drop_collinear(X_tr, X_te)
    if dropped:
        print(f"    dropped near-collinear columns: {sorted(dropped)}")

    df_tr = X_tr.copy()
    df_tr["__label__"] = y_tr.values if hasattr(y_tr, "values") else y_tr
    n_train = len(df_tr)

    df_disc = df_tr.copy()
    bin_edges = {}
    for c in df_tr.columns[:-1]:
        edges = np.linspace(0, 1, 11)
        df_disc[c] = np.digitize(df_tr[c].values, edges[1:-1])
        bin_edges[c] = edges

    extra_kwargs = {}
    timeout = MST_TIMEOUT_SEC
    if synth_name == "dpctgan":
        if n_train > DPCTGAN_MAX_ROWS:
            return None, f"SKIP (dpctgan): n_train={n_train} > {DPCTGAN_MAX_ROWS}, using MST only for this dataset"
        batch_size = max(8, min(64, n_train // 4))
        batch_size -= batch_size % 2
        batch_size = max(batch_size, 8)
        extra_kwargs = dict(batch_size=batch_size, epochs=30, discriminator_steps=1)
        timeout = DPCTGAN_TIMEOUT_SEC
    elif synth_name == "aim":
        # AIM can be slow on high-dimensional data; reuse the same timeout as MST
        timeout = MST_TIMEOUT_SEC

    synth = Synthesizer.create(synth_name, epsilon=eps, verbose=False, **extra_kwargs)
    t0 = time.time()

    def _fit_and_sample():
        # preprocessor_eps=0.0 is legitimate now because X_tr was scaled using
        # public bounds (public_bounds_for). No private bounds are learned.
        synth.fit(df_disc, preprocessor_eps=0.0)
        return synth.sample(len(df_disc))

    try:
        sample = run_with_timeout(_fit_and_sample, timeout)
    except TimeoutException:
        return None, f"SKIP ({synth_name}): timed out after {timeout}s (likely degenerate/collinear marginals)"
    except Exception as e:
        return None, f"SKIP ({synth_name}): {e}"
    elapsed = time.time() - t0

    X_synth = pd.DataFrame(index=range(len(sample)))
    for c in df_tr.columns[:-1]:
        centers = (bin_edges[c][:-1] + bin_edges[c][1:]) / 2
        X_synth[c] = centers[np.clip(sample[c].values.astype(int), 0, 9)]
    y_synth = sample["__label__"].values
    if len(np.unique(y_synth)) < 2:
        return None, f"SKIP ({synth_name}): synthetic set collapsed to one class"

    util, model = evaluate_utility(X_synth, pd.Series(y_synth), X_te, y_te)

    # FIX 6: Score the REAL train/test data through the model trained on synthetic data.
    # This matches the protocol used for DP-GBC/DPLC in Cell 7.
    conf_tr = np.max(model.predict_proba(X_tr), axis=1)
    conf_te = np.max(model.predict_proba(X_te), axis=1)
    scores = np.concatenate([conf_tr, conf_te])
    labels = np.concatenate([np.ones(len(conf_tr)), np.zeros(len(conf_te))])
    mia = roc_auc_score(labels, scores)
    return {"bal_acc": util["balanced_accuracy"], "mia_rf": mia, "fit_seconds": elapsed}, None

# Precompute public bounds once per dataset (no privacy budget spent)
bounds_cache = {}
for ds_name, (X, y) in datasets.items():
    bounds_cache[ds_name] = public_bounds_for(ds_name, X)

synth_results = []
for ds_name, (X, y) in datasets.items():
    lo, hi = bounds_cache[ds_name]   # public bounds
    for seed in SEEDS[:2]:
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
        except ValueError:
            continue

        # Public-bounds scaling (replaces private MinMaxScaler)
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
        X_te_n = scale_with_public_bounds(X_te, lo, hi)

        # ---- Fairness patch: also score DP-GBC on the SAME reduced feature
        # set each synthesizer actually saw, so df_synth is an apples-to-apples
        # table on its own (df_main's dpgbc_bal_acc used the FULL feature set
        # and is not comparable to these rows).
        _, _, dropped_probe = drop_collinear(X_tr_n, X_te_n)
        if dropped_probe:
            keep_mask = [c not in dropped_probe for c in X.columns]
            X_r = X.loc[:, keep_mask]
            lo_r, hi_r = lo[keep_mask], hi[keep_mask]
            max_depth_r = 3
            eps_cnt_r = eps_spl_r = 0.1
            eps_rel_r = TOTAL_EPS - max_depth_r * (eps_cnt_r + eps_spl_r)
            X_tr_r, X_te_r, y_tr_r, y_te_r, ledger_r, _, balls_r = run_dp_gbc_pipeline(
                X_r, y, seed, lo_r, hi_r, max_depth=max_depth_r,
                eps_count_per_level=eps_cnt_r, eps_split_per_level=eps_spl_r,
                eps_center_per_level=0.0, eps_radius_per_level=0.0,
                eps_count_release=eps_rel_r,
                all_classes=classes_public[ds_name],
                precomputed_split=(X_r.loc[X_tr.index], X_r.loc[X_te.index], y_tr, y_te)
            )
            assert abs(ledger_r.total_eps - TOTAL_EPS) < 1e-6
            preds_r = dp_gbc_classify_predict(balls_r, X_te_r)
            bal_acc_r = balanced_accuracy_score(y_te_r, preds_r)
            mia_r, _ = membership_inference_attack_rf_synth(
                X_tr_r, y_tr_r, X_tr_n[[c for c in X_tr_n.columns if c not in dropped_probe]], y_tr,
                X_te_n[[c for c in X_te_n.columns if c not in dropped_probe]], y_te, seed
            )
            synth_results.append({"Dataset": ds_name, "Seed": seed, "method": "dpgbc_same_features",
                                   "bal_acc": bal_acc_r, "mia_rf": mia_r, "fit_seconds": np.nan})

        # FIX 4: include AIM alongside MST and DP-CTGAN
        for synth_name in ["mst", "aim", "dpctgan"]:
            t_start = time.time()
            res, err = run_synth_baseline(X_tr_n, y_tr, X_te_n, y_te, TOTAL_EPS, synth_name, seed)
            if res is None:
                print(f"{ds_name}/{seed}/{synth_name}: {err}")
                continue
            print(f"{ds_name}/{seed}/{synth_name}: done in {time.time()-t_start:.1f}s")
            synth_results.append({"Dataset": ds_name, "Seed": seed, "method": synth_name, **res})

df_synth = pd.DataFrame(synth_results)
if len(df_synth):
    print(df_synth.groupby(["Dataset","method"])[["bal_acc","mia_rf"]].mean().round(3).to_string())
else:
    print("No rows produced -- check snsynth install / dataset sizes.")

In [ ]:
# Cell 19 (FIXED): DP-PCA baseline (input-perturbation family: BDMS05 / AGTZ14)
# Bug 1: row clipping now shrinks over-norm rows (norm ≤ 1) instead of normalizing all rows.
# Bug 2: Laplace noise on projected Z now correctly uses L1 sensitivity sqrt(k).
# FIX (public bounds): private MinMaxScaler removed; use public_bounds_for.

def dp_pca_transform(X_tr, X_te, eps, n_components, rng):
    Xc = X_tr.values - 0.5   # fixed PUBLIC center (data is [0,1] → midpoint 0.5)
    norms = np.linalg.norm(Xc, axis=1, keepdims=True)
    # Bug 1 fix: clip rows whose L2 norm > 1 to unit norm; leave others untouched
    clip_scale = np.minimum(1.0, 1.0 / np.maximum(norms, 1e-8))
    Xc_unit = Xc * clip_scale
    A = Xc_unit.T @ Xc_unit
    d = A.shape[0]
    noise = rng.normal(0, 2.0 * np.sqrt(2*np.log(1.25/1e-5)) / max(eps, 1e-6), size=(d, d))
    noise = np.triu(noise); noise = noise + noise.T - np.diag(np.diag(noise))
    A_noisy = (A + noise); A_noisy = (A_noisy + A_noisy.T) / 2
    eigvals, eigvecs = np.linalg.eigh(A_noisy)
    proj = eigvecs[:, np.argsort(eigvals)[::-1][:n_components]]
    Xte_c = X_te.values - 0.5
    return Xc @ proj, Xte_c @ proj

dppca_results = []
for ds_name, (X, y) in datasets.items():
    # Public bounds for this dataset (no privacy budget spent)
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
        except ValueError:
            continue

        # Public-bounds scaling (replaces private MinMaxScaler)
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
        X_te_n = scale_with_public_bounds(X_te, lo, hi)

        eps_pca = TOTAL_EPS * 0.4
        eps_noise = TOTAL_EPS - eps_pca
        k = max(1, min(5, X.shape[1] - 1))
        Z_tr, Z_te = dp_pca_transform(X_tr_n, X_te_n, eps_pca, k, rng)

        # L2 sensitivity = 1 — code is already correct; the previous comment was wrong.
        delta_z = 1e-5
        sigma_z = 1.0 * np.sqrt(2 * np.log(1.25 / delta_z)) / max(eps_noise, 1e-6)  # L2 sensitivity = 1
        Z_tr = np.clip(Z_tr + rng.normal(0, sigma_z, Z_tr.shape), -1, 1)
        Z_te = np.clip(Z_te, -1, 1)   # test set isn't part of the private release -- don't noise it

        Z_tr_df = pd.DataFrame(Z_tr, columns=[f"pc{i}" for i in range(Z_tr.shape[1])])
        Z_te_df = pd.DataFrame(Z_te, columns=[f"pc{i}" for i in range(Z_te.shape[1])])
        util, _ = evaluate_utility(Z_tr_df, y_tr.reset_index(drop=True), Z_te_df, y_te.reset_index(drop=True))
        mia, _ = membership_inference_attack_rf(Z_tr_df, y_tr.reset_index(drop=True), Z_te_df, y_te.reset_index(drop=True), seed)
        dppca_results.append({"Dataset": ds_name, "Seed": seed,
                               "dppca_bal_acc": util["balanced_accuracy"], "dppca_mia_rf": mia})

print(pd.DataFrame(dppca_results).groupby("Dataset")[["dppca_bal_acc","dppca_mia_rf"]].mean().round(3).to_string())

In [ ]:
# Cell 20 (NEW): DP-SGD / gradient-perturbation baseline (Abadi et al. 2016;
# Chaudhuri, Monteleoni, Sarwate 2011). We implement FULL-BATCH DP-GD, not the
# Poisson-subsampled minibatch version -- subsampling amplification needs a
# moments accountant / RDP subsampling theorem that's easy to misstate from
# memory. Full-batch DP-GD has clean, verifiable zCDP accounting: each
# iteration is a Gaussian release of a clipped-and-averaged gradient with
# sensitivity C/n, so rho_per_iter = 1/(2*sigma^2) zCDP (sigma = noise
# multiplier), additive over T iterations, converted via Bun & Steinke 2016's
# zCDP->DP formula. Binary classification only (noted below).

def zcdp_to_eps(rho, delta):
    return rho + 2 * np.sqrt(rho * np.log(1 / delta))

def dp_class_weights(y_tr, eps_count, rng):
    classes = np.unique(y_tr)
    n = len(y_tr)
    noisy_counts = {}
    for c in classes:
        true_count = np.sum(y_tr == c)
        noisy_counts[c] = max(1.0, true_count + rng.laplace(0, 1.0 / max(eps_count, 1e-6)))
    return {c: n / (len(classes) * cnt) for c, cnt in noisy_counts.items()}

def dp_sgd_logreg(X_tr, y_tr, eps_target, delta, T=100, clip_norm=1.0, lr=0.5, rng=None,
                   max_class_weight=3.0, eps_count_for_weights=0.05):
    rng = rng or np.random.default_rng(0)
    n = X_tr.shape[0]
    Xb = np.hstack([X_tr, np.ones((n, 1))])
    w = np.zeros(Xb.shape[1])
    y_pm = np.where(y_tr == y_tr.min(), -1, 1)

    # class weights now DP-released, spending a small slice of the budget
    eps_target_gd = eps_target - eps_count_for_weights  # remaining budget for the actual training
    raw_weight = dp_class_weights(y_tr, eps_count_for_weights, rng)
    sample_w = np.array([raw_weight[v] for v in y_tr])
    sample_w = np.clip(sample_w / sample_w.mean(), 0.0, max_class_weight)
    effective_clip = clip_norm * max_class_weight

    lo, hi = 0.1, 100.0
    for _ in range(40):
        mid = (lo + hi) / 2
        rho = T / (2 * mid**2)
        if zcdp_to_eps(rho, delta) > eps_target_gd: lo = mid
        else: hi = mid
    sigma = hi

    for t in range(T):
        p = 1 / (1 + np.exp(-y_pm * (Xb @ w)))
        grad_i = (-(1 - p) * y_pm)[:, None] * Xb                     # raw gradient, UNWEIGHTED
        norms = np.linalg.norm(grad_i, axis=1, keepdims=True)
        grad_i = grad_i * np.minimum(1.0, clip_norm / np.maximum(norms, 1e-8))  # clip FIRST
        grad_avg = (sample_w[:, None] * grad_i).mean(axis=0)          # weight applied in the AVERAGE
        w = w - lr * (grad_avg + rng.normal(0, effective_clip * sigma / n, size=grad_avg.shape))
    return w, sigma

def dp_sgd_predict_proba(w, X):
    Xb = np.hstack([X, np.ones((X.shape[0], 1))])
    p1 = 1 / (1 + np.exp(-(Xb @ w)))
    return np.stack([1 - p1, p1], axis=1)

dpsgd_results = []
for ds_name, (X, y) in datasets.items():
    if y.nunique() != 2:
        continue

    # Public bounds for this dataset (no privacy budget spent)
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
        except ValueError:
            continue

        # Public-bounds scaling (replaces private MinMaxScaler)
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
        X_te_n = scale_with_public_bounds(X_te, lo, hi)

        # CHANGE: T from 100 -> 25 (5th positional argument)
        w, sigma_used = dp_sgd_logreg(X_tr_n.values, y_tr.values, TOTAL_EPS, 1e-5, 100, 1.0, 0.5, rng)
        preds = dp_sgd_predict_proba(w, X_te_n.values).argmax(axis=1) + y_tr.min()
        bal_acc = balanced_accuracy_score(y_te, preds)
        if bal_acc == 0.5:
            print(f"{ds_name}: w norm = {np.linalg.norm(w):.4f}, "
                  f"noise std per step = {1.0 * sigma_used / len(X_tr_n):.4f}")

        conf_tr = np.max(dp_sgd_predict_proba(w, X_tr_n.values), axis=1)
        conf_te = np.max(dp_sgd_predict_proba(w, X_te_n.values), axis=1)
        try:
            mia = roc_auc_score(np.concatenate([np.ones(len(conf_tr)), np.zeros(len(conf_te))]),
                                 np.concatenate([conf_tr, conf_te]))
        except Exception:
            mia = 0.5
        dpsgd_results.append({"Dataset": ds_name, "Seed": seed, "dpsgd_bal_acc": bal_acc,
                               "dpsgd_mia": mia, "sigma_used": sigma_used})

n_multiclass = sum(1 for _, (_, y) in datasets.items() if y.nunique() != 2)
# ADD: include sigma_used in the printed summary and assign dataframe to variable
df_dpsgd_out = pd.DataFrame(dpsgd_results)
print(df_dpsgd_out.groupby("Dataset")[["dpsgd_bal_acc","dpsgd_mia","sigma_used"]].mean().round(3).to_string())
print(f"\nbinary-only baseline -- {n_multiclass} multiclass datasets excluded, state this in limitations")
print("DP-SGD-LR baseline: T=100 resolves the earlier BankMarketing iteration-limit "
      "(0.500 -> 0.627). CreditFraud remains at chance even at T=100 with near-zero "
      "noise (isolation test), consistent with its extreme 500:1 class imbalance "
      "exceeding what max_class_weight=3.0 can compensate for -- a separate, unresolved "
      "limitation of this baseline, not an iteration-count artifact.")

In [ ]:
# Cell 21 (FIXED): DP ensemble-of-trees, standing in for "DP-GBDT".
# Corrected per review:
#   - uses dp_generate_granular_balls_with_radius (public box bounds)
#   - uses DP-released noisy_class_counts for leaf labels
#   - public schema bounds for scaling, not private MinMaxScaler

def dp_random_forest_predict(X_tr, y_tr, X_te, n_trees, total_eps, max_depth, min_size, rng, all_classes):
    eps_per_tree = total_eps / n_trees
    eps_count = eps_split = eps_per_tree / (2 * max_depth)   # only count+split are charged, matches ledger elsewhere
    votes = np.zeros((len(X_te), len(all_classes)))
    from scipy.spatial.distance import cdist
    for t in range(n_trees):
        feat_subset = rng.choice(X_tr.shape[1], size=max(1, int(np.sqrt(X_tr.shape[1]))), replace=False)
        balls = dp_generate_granular_balls_with_radius(   # <-- box-tracked, public-threshold version
            pd.DataFrame(X_tr[:, feat_subset]), y_tr, max_depth=max_depth,
            eps_count_per_level=eps_count, eps_split_per_level=eps_split,
            eps_radius_per_level=0.0, eps_center_per_level=0.0,
            min_size=min_size, rng=rng, all_classes=all_classes)
        centers = np.array([b["center"] for b in balls])
        leaf_labels = np.array([
            b["classes"][np.argmax(np.clip(b["noisy_class_counts"], 0, None))]   # <-- use the DP counts
            for b in balls
        ])
        assign = np.argmin(cdist(X_te[:, feat_subset], centers), axis=1)
        for i, a in enumerate(assign):
            votes[i, int(leaf_labels[a])] += 1
    return votes.argmax(axis=1)

dprf_results = []
for ds_name, (X, y) in datasets.items():
    # Public bounds for this dataset (no privacy budget spent)
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
        except ValueError:
            continue

        # Public-bounds scaling (replaces private MinMaxScaler)
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi).values
        X_te_n = scale_with_public_bounds(X_te, lo, hi).values

        preds = dp_random_forest_predict(
            X_tr_n, y_tr.values, X_te_n, 5, TOTAL_EPS, 3, 10, rng,
            all_classes=classes_public[ds_name]
        )
        dprf_results.append({"Dataset": ds_name, "Seed": seed,
                              "dprf_bal_acc": balanced_accuracy_score(y_te, preds)})

print(pd.DataFrame(dprf_results).groupby("Dataset")["dprf_bal_acc"].mean().round(3).to_string())

In [ ]:
# Cell 22 (FIXED): Reconstruction attack.
# Uses public schema bounds for scaling, not private MinMaxScaler, to avoid
# leaking bounds twice and inflating the attack_improvement metric.

from sklearn.neural_network import MLPRegressor

recon_results = []
for ds_name, (X, y) in datasets.items():
    # Public bounds for this dataset (no privacy budget spent)
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=seed, stratify=y)
        except ValueError:
            continue
        X_priv, X_aux = train_test_split(X_tr, test_size=0.5, random_state=seed)

        # Public-bounds scaling (replaces MinMaxScaler().fit(X_tr))
        X_priv_n = scale_with_public_bounds(X_priv, lo, hi)
        X_aux_n = scale_with_public_bounds(X_aux, lo, hi)

        rng2 = np.random.default_rng(seed + 1)
        X_priv_flat = np.clip(X_priv_n + rng2.laplace(0, 1.0 / TOTAL_EPS, X_priv_n.shape), 0, 1)
        X_aux_flat = np.clip(X_aux_n + rng2.laplace(0, 1.0 / TOTAL_EPS, X_aux_n.shape), 0, 1)

        denoiser = MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=500, random_state=0)
        denoiser.fit(X_aux_flat, X_aux_n)
        X_recon = denoiser.predict(X_priv_flat)

        mse_recon = np.mean((X_recon - X_priv_n) ** 2)
        mse_naive = np.mean((X_priv_flat - X_priv_n) ** 2)
        recon_results.append({"Dataset": ds_name, "Seed": seed, "mse_reconstructed": mse_recon,
                               "mse_naive_noised": mse_naive, "attack_improvement": mse_naive - mse_recon})

df_recon = pd.DataFrame(recon_results)
print(df_recon.groupby("Dataset")[["mse_naive_noised","mse_reconstructed","attack_improvement"]].mean().round(4).to_string())
print("\npositive attack_improvement = denoiser meaningfully beat raw noised values (privacy concern)")

In [ ]:
# Cell 23 (UPDATED): Offline population-calibrated MIA (Carlini et al., IEEE S&P 2022, LiRA).
#
# Fixes:
#   - public schema bounds instead of private dp_release_feature_bounds
#   - updated run_dp_localclip_pipeline signature with lo, hi
#   - explicit eps_center_per_level
#   - budget still sums to TOTAL_EPS
#
# NOTE (CAVEAT): The shadow models in offline_lira are trained with flat Laplace noise,
# but the target model here is trained on the DP-LocalClip synthetic release.
# This violates LiRA's assumption that shadow and target models use the same mechanism.
# This exploratory cell is retained as-is for a rough estimate only; do not use
# these results as a rigorous privacy claim without rewriting offline_lira to
# train shadow models via run_dp_localclip_pipeline per shadow trial.

def offline_lira(X_pool, y_pool, target_idx, n_shadow, eps, rng):
    n = len(X_pool)
    out_conf = {i: [] for i in target_idx}
    for s in range(n_shadow):
        rs = np.random.default_rng(rng.integers(1e9))
        keep = rs.choice(n, size=n // 2, replace=False)
        keep_set = set(keep.tolist())
        X_shadow, y_shadow = X_pool[keep], y_pool[keep]
        X_shadow_flat = np.clip(X_shadow + rs.laplace(0, 1.0 / eps, X_shadow.shape), 0, 1)
        if len(np.unique(y_shadow)) < 2:
            continue
        model = LogisticRegression(max_iter=500, class_weight="balanced", random_state=s)
        model.fit(X_shadow_flat, y_shadow)
        for i in target_idx:
            if i not in keep_set:
                x_n = np.clip(
                    X_pool[i:i + 1] + rs.laplace(0, 1.0 / eps, X_pool[i:i + 1].shape),
                    0, 1
                )
                conf = model.predict_proba(x_n)[0, y_pool[i]]
                out_conf[i].append(np.log(conf / max(1 - conf, 1e-8)))
    return out_conf


TOTAL_EPS = float(globals().get("TOTAL_EPS", 3.0))

# Optional numeric-only filter (same as main results)
NUMERIC_DATASETS = [
    "BreastCancer",
    "PimaDiabetes",
    "BloodTransfusion",
    "MAGIC04",
    "Banknote",
    "Ionosphere",
    "Sonar",
    "Wine",
    "WisconsinOriginal",
    "Waveform",
    "Vehicle",
    "Segment",
    "Spambase",
    "WineQuality",
]

lira_results = []

for ds_name, (X, y) in datasets.items():
    if ds_name not in NUMERIC_DATASETS:
        continue

    seed = SEEDS[0]
    rng = np.random.default_rng(seed)

    try:
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.3, random_state=seed, stratify=y
        )
    except ValueError:
        continue

    # Public schema bounds (no privacy budget spent)
    lo, hi = public_bounds_for(ds_name, X)

    # Public-bounds scaling — no private MinMax
    X_tr_n = scale_with_public_bounds(X_tr, lo, hi).values
    X_te_n = scale_with_public_bounds(X_te, lo, hi)
    y_tr_arr = y_tr.values

    target_idx = rng.choice(
        len(X_tr_n), size=min(40, len(X_tr_n)), replace=False
    )

    # Offline LiRA shadow-model phase
    out_conf = offline_lira(
        X_tr_n,
        y_tr_arr,
        target_idx,
        n_shadow=16,
        eps=TOTAL_EPS,
        rng=rng
    )

    # Fixed DP-LocalClip synthetic release
    eps_cnt = eps_spl = 0.1
    eps_ctr = 0.0   # public
    eps_rad = 0.0   # public
    eps_noise_d = TOTAL_EPS - 3 * (eps_cnt + eps_spl)

    X_tr_d, X_te_d, y_tr_d, y_te_d, _, _ = run_dp_localclip_pipeline(
        X, y, seed, lo, hi,
        max_depth=3,
        eps_count_per_level=eps_cnt,
        eps_split_per_level=eps_spl,
        eps_radius_per_level=eps_rad,
        eps_center_per_level=eps_ctr,
        eps_noise=eps_noise_d,
        all_classes=classes_public[ds_name]
    )

    if len(X_tr_d) == 0:
        continue

    _, target_model = evaluate_utility(X_tr_d, y_tr_d, X_te_d, y_te_d)

    correct, valid = 0, 0
    for i in target_idx:
        if len(out_conf[i]) < 3:
            continue
        mu = np.mean(out_conf[i])
        sd = np.std(out_conf[i]) + 1e-6

        # Original training point, public-bounds scaled, not noised
        obs_conf = target_model.predict_proba(X_tr_n[i:i + 1])[0, y_tr_arr[i]]
        obs_logit = np.log(obs_conf / max(1 - obs_conf, 1e-8))

        valid += 1
        if (obs_logit - mu) / sd > 0:
            correct += 1

    if valid:
        lira_results.append({
            "Dataset": ds_name,
            "n_targets": valid,
            "guess_member_rate": correct / valid,
        })

print(pd.DataFrame(lira_results).round(3).to_string(index=False))
print("\nguess_member_rate near 0.5 = attack can't distinguish member vs non-member")

In [ ]:
# Cell 24 (FIXED): Property inference attack
# Uses public schema bounds for scaling, not private MinMaxScaler.

from sklearn.linear_model import LinearRegression

def property_inference_attack(X, y, ds_name, eps, n_shadow, rng):
    X_df = X if hasattr(X, "values") else pd.DataFrame(X)
    y_arr = y.values if hasattr(y, "values") else np.asarray(y)
    # Public bounds from schema; no privacy budget spent
    lo, hi = public_bounds_for(ds_name, X_df)
    X_n = scale_with_public_bounds(X_df, lo, hi).values

    probe = X_n[rng.choice(len(X_n), size=min(50, len(X_n)), replace=False)]
    pos_val = y_arr.max()
    meta_X, meta_y = [], []
    for s in range(n_shadow):
        rs = np.random.default_rng(rng.integers(1e9))
        target_frac = rs.uniform(0.1, 0.9)
        pos_idx = np.where(y_arr == pos_val)[0]; neg_idx = np.where(y_arr != pos_val)[0]
        n_take = min(len(pos_idx), len(neg_idx), 200)
        n_pos = int(n_take * target_frac); n_neg = n_take - n_pos
        if n_pos < 2 or n_neg < 2: continue
        idx = np.concatenate([rs.choice(pos_idx, n_pos, replace=True), rs.choice(neg_idx, n_neg, replace=True)])
        X_shadow_flat = np.clip(X_n[idx] + rs.laplace(0, 1.0 / eps, (len(idx), X_n.shape[1])), 0, 1)
        model = LogisticRegression(max_iter=500, class_weight="balanced", random_state=s)
        model.fit(X_shadow_flat, y_arr[idx])
        probs = model.predict_proba(probe)
        meta_X.append([probs.mean(), probs.std(), np.percentile(probs[:, -1], 25), np.percentile(probs[:, -1], 75)])
        meta_y.append(target_frac)
    if len(meta_X) < 10: return None
    meta_X, meta_y = np.array(meta_X), np.array(meta_y)
    split = int(0.7 * len(meta_X))
    meta_model = LinearRegression().fit(meta_X[:split], meta_y[:split])
    mae = np.mean(np.abs(meta_model.predict(meta_X[split:]) - meta_y[split:]))
    baseline_mae = np.mean(np.abs(meta_y[split:] - meta_y[:split].mean()))
    return {"Dataset": ds_name, "attack_mae": mae, "baseline_mae": baseline_mae}

pia_results = []
rng = np.random.default_rng(SEED)
for ds_name, (X, y) in datasets.items():
    if y.nunique() != 2: continue
    res = property_inference_attack(X, y, ds_name, TOTAL_EPS, 60, rng)
    if res: pia_results.append(res)

df_pia = pd.DataFrame(pia_results)
df_pia["attack_beats_baseline"] = df_pia["attack_mae"] < df_pia["baseline_mae"]
print(df_pia.round(3).to_string(index=False))

In [ ]:
# Cell 25 (REVISED): Bayesian AIA across flat, DP-GBC, and DP-LocalClip.
# Public bounds used instead of private DP-released bounds.

from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import KBinsDiscretizer

# Sensitive column mapping, same as Cell 7
SENSITIVE_COL = {
    "AdultIncome": "age",
    "BreastCancer": "mean radius",
    "PimaDiabetes": "age",
    "HeartHungarian": "age",
    "CreditG": "age",
}

def resolve_sens_col_idx(X, ds_name):
    target = SENSITIVE_COL.get(ds_name)
    if target is None:
        return 0
    cols_lower = [c.lower() for c in X.columns]
    if target.lower() in cols_lower:
        return cols_lower.index(target.lower())
    print(f"WARNING: sensitive column '{target}' not found for {ds_name}; falling back to col 0")
    return 0


def bayesian_aia(X_tr_attack, X_te_attack, sens_tr, sens_te):
    """
    Same function as before. Returns accuracy - majority-class baseline.
    """
    sens_tr = np.nan_to_num(np.asarray(sens_tr).ravel())
    sens_te = np.nan_to_num(np.asarray(sens_te).ravel())

    if len(np.unique(sens_tr)) > 10:
        disc = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile")
        sens_tr = disc.fit_transform(sens_tr.reshape(-1, 1)).ravel().astype(int)
        sens_te = disc.transform(sens_te.reshape(-1, 1)).ravel().astype(int)
    else:
        sens_tr = sens_tr.astype(int)
        sens_te = sens_te.astype(int)

    if len(np.unique(sens_tr)) < 2:
        return 0.0

    clf = GaussianNB().fit(X_tr_attack, sens_tr)
    preds = clf.predict(X_te_attack)
    baseline = pd.Series(sens_te).value_counts(normalize=True).max()
    return accuracy_score(sens_te, preds) - baseline


TOTAL_EPS = float(globals().get("TOTAL_EPS", 3.0))

bayes_aia_results = []

for ds_name, (X, y) in datasets.items():
    if y.nunique() != 2:
        continue

    sens_col_idx = resolve_sens_col_idx(X, ds_name)
    sens_col_name = X.columns[sens_col_idx]

    # Public bounds from schema; no privacy budget spent
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.3, random_state=seed, stratify=y
            )
        except ValueError:
            continue

        # Public-bounds scaling — not private MinMax
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
        X_te_n = scale_with_public_bounds(X_te, lo, hi)

        non_sens_cols = [c for c in X.columns if c != sens_col_name]

        # ------------------- Flat noise -------------------
        rng = np.random.default_rng(seed)
        X_tr_flat = make_flat_noised(X_tr_n, TOTAL_EPS, rng)
        X_te_flat = make_flat_noised(X_te_n, TOTAL_EPS, rng)

        flat_lift = bayesian_aia(
            X_tr_flat[non_sens_cols].values,
            X_te_flat[non_sens_cols].values,
            X_tr_n[sens_col_name].values,   # original scaled sensitive, not noised
            X_te_n[sens_col_name].values    # original scaled sensitive, not noised
        )

        # ------------------- DP-GBC -------------------
        max_depth = 3
        eps_count = eps_split = 0.1
        eps_center = 0.0   # public
        eps_radius = 0.0   # public
        eps_count_release = TOTAL_EPS - max_depth * (eps_count + eps_split)

        X_tr_c, X_te_c, y_tr_c, y_te_c, ledger_c, _, balls_c = run_dp_gbc_pipeline(
            X, y, seed, lo, hi,          # FIX: was seed * 2
            max_depth=max_depth,
            eps_count_per_level=eps_count,
            eps_split_per_level=eps_split,
            eps_center_per_level=eps_center,
            eps_radius_per_level=eps_radius,
            eps_count_release=eps_count_release,
            all_classes=classes_public[ds_name]
        )

        dpgbc_lift = bayesian_aia(
            X_tr_c[non_sens_cols].values,
            X_te_c[non_sens_cols].values,
            X_tr_c[sens_col_name].values,   # synthetic sensitive, used as training target
            X_te_c[sens_col_name].values    # real scaled sensitive, used as test target
        )

        # ------------------- DP-LocalClip -------------------
        eps_noise_lc = eps_count_release  # same remaining budget after fixed redesign

        X_tr_lc, X_te_lc, y_tr_lc, y_te_lc, ledger_lc, _ = run_dp_localclip_pipeline(
            X, y, seed, lo, hi,          # FIX: was seed * 2 + 1
            max_depth=max_depth,
            eps_count_per_level=eps_count,
            eps_split_per_level=eps_split,
            eps_radius_per_level=eps_radius,
            eps_center_per_level=eps_center,
            eps_noise=eps_noise_lc,
            all_classes=classes_public[ds_name]
        )

        dplc_lift = bayesian_aia(
            X_tr_lc[non_sens_cols].values,
            X_te_lc[non_sens_cols].values,
            X_tr_lc[sens_col_name].values,
            X_te_lc[sens_col_name].values
        )

        bayes_aia_results.append({
            "Dataset": ds_name,
            "Seed": seed,
            "flat_bayes_aia_lift": flat_lift,
            "dpgbc_bayes_aia_lift": dpgbc_lift,
            "dplc_bayes_aia_lift": dplc_lift,
        })

# Per-dataset means
summary = pd.DataFrame(bayes_aia_results).groupby("Dataset")[
    ["flat_bayes_aia_lift", "dpgbc_bayes_aia_lift", "dplc_bayes_aia_lift"]
].mean().round(3)

print(summary.to_string())

In [ ]:
# Cell 26 (MODIFIED v7): empirical canary lower-bound audit — **DISCLAIMED**
#
# ⚠️ IMPORTANT: This audit is retained for transparency only.
# The positive control (eps_config=1e6) shows zero detection power on most
# datasets, meaning the canary/statistic combination is NOT sensitive enough
# to detect membership even in the noiseless limit. Therefore the resulting
# "eps_lower_bound" values MUST NOT be interpreted as evidence of differential
# privacy or as a lower bound on the true privacy loss.
#
# This cell is kept to document the attempted audit and its limitation.
# It should not be used in the paper to support privacy claims.
#
# ----------------------------------------------------------------------------
# The audit itself is identical to previous versions, but the final output
# includes an explicit warning and the positive control is emphasised.

from scipy.stats import beta as beta_dist
from scipy.spatial.distance import cdist
import numpy as np
import pandas as pd


# ----------------------------------------------------------------------------
# Clopper-Pearson bounds
# ----------------------------------------------------------------------------
def clopper_pearson(k, n, alpha=0.05):
    """Exact binomial confidence interval bounds."""
    if n <= 0:
        return 0.0, 0.0
    lo = 0.0 if k == 0 else beta_dist.ppf(alpha / 2, k, n - k + 1)
    hi = 1.0 if k == n else beta_dist.ppf(1 - alpha / 2, k + 1, n - k)
    return float(lo), float(hi)


# ----------------------------------------------------------------------------
# Fixed DP synthetic release WITHOUT train/test splitting.
# Assumes X_scaled_df is already scaled to [0, 1] using public bounds.
# ----------------------------------------------------------------------------
def run_dp_synthetic_release_no_split_scaled(
    X_scaled_df,
    y_series,
    seed,
    max_depth,
    eps_count_per_level,
    eps_split_per_level,
    eps_radius_per_level,
    eps_center_per_level,
    eps_count_release,
    delta=0.0,               # <-- Fix 5: pure (eps,0)-DP, no delta
    all_classes=None
):
    """
    Same fixed synthetic release mechanism as the main pipeline, but:
      1. No train/test split.
      2. Input is already publicly scaled.
      3. Returns a synthetic dataset; row order/size is not preserved.
    """
    rng = np.random.default_rng(seed)
    ledger = PrivacyLedger()

    X_n = X_scaled_df.copy().reset_index(drop=True)
    y_n = y_series.copy().reset_index(drop=True)

    balls = dp_generate_granular_balls_with_radius(
        X_n,
        y_n.values,
        max_depth=max_depth,
        eps_count_per_level=eps_count_per_level,
        eps_split_per_level=eps_split_per_level,
        eps_radius_per_level=eps_radius_per_level,
        eps_center_per_level=eps_center_per_level,
        rng=rng,
        all_classes=all_classes
    )

    per_level = (
        eps_count_per_level +
        eps_split_per_level +
        eps_radius_per_level +
        eps_center_per_level
    )
    for depth in range(max_depth):
        ledger.spend_sequential_charged_once(
            per_level,
            delta=delta,               # now delta=0.0 by default
            n_branches=len(balls),
            note=f"granulation depth {depth+1}"
        )

    ledger.spend_sequential(
        eps_count_release,
        0.0,
        note="per-ball synthetic-size release (pure-eps)"
    )
    sizes = dp_release_ball_counts(balls, eps_count_release, rng)
    X_synth, y_synth = dp_sample_synthetic_records(
        balls, sizes, rng, d=X_n.shape[1]
    )

    return (
        pd.DataFrame(X_synth, columns=X_n.columns),
        pd.Series(y_synth),
        ledger
    )


# ----------------------------------------------------------------------------
# Trial construction helpers
# ----------------------------------------------------------------------------
def _make_corner_canary_trial(
    X_arr,
    y_arr,
    rng,
    trial_size,
    canary_x,
    canary_y,
    class_indices,
    classes
):
    """Construct one fixed-size audit trial."""
    n = len(X_arr)
    trial_size = int(min(trial_size, n))

    if trial_size < 2:
        raise ValueError("trial_size must be at least 2.")

    base_size = trial_size - 1
    base_idx = rng.choice(n, size=base_size, replace=False)

    remaining = np.setdiff1d(np.arange(n), base_idx, assume_unique=False)
    if len(remaining) == 0:
        placebo_idx = int(base_idx[0])
    else:
        placebo_idx = int(rng.choice(remaining))

    include = bool(rng.random() < 0.5)

    if include:
        X_in = np.vstack([
            X_arr[base_idx],
            canary_x.reshape(1, -1)
        ])
        y_in = np.concatenate([
            y_arr[base_idx],
            np.array([canary_y])
        ])
    else:
        X_in = np.vstack([
            X_arr[base_idx],
            X_arr[placebo_idx].reshape(1, -1)
        ])
        y_in = np.concatenate([
            y_arr[base_idx],
            np.array([y_arr[placebo_idx]])
        ])

    if len(classes) > 1 and len(np.unique(y_in)) == 1:
        current_label = int(y_in[0])
        other_labels = [int(c) for c in classes if int(c) != current_label]
        if len(other_labels) > 0:
            other_label = int(rng.choice(other_labels))
            forced_idx = int(rng.choice(class_indices[other_label]))
            X_in[0] = X_arr[forced_idx]
            y_in[0] = y_arr[forced_idx]

    return include, X_in, y_in


# ----------------------------------------------------------------------------
# Core audit runner
# ----------------------------------------------------------------------------
def _run_canary_audit_core(
    X_scaled,
    y,
    eps_config,
    n_trials,
    seed,
    trial_size=200
):
    X_arr = np.asarray(X_scaled, dtype=float)
    y_arr = np.asarray(y).ravel()

    n, d = X_arr.shape

    if n < 4 or eps_config <= 0:
        return (
            np.array([], dtype=float),
            np.array([], dtype=bool),
            np.array([], dtype=float)
        )

    classes, counts = np.unique(y_arr, return_counts=True)
    class_indices = {
        int(c): np.where(y_arr == c)[0]
        for c in classes
    }

    canary_y = int(classes[np.argmax(counts)])
    canary_x = np.ones(d, dtype=float)

    max_depth = 3

    granulation_target = 0.7 * float(eps_config)
    count_release = float(eps_config) - granulation_target

    unit_per_level = 0.1 + 0.1   # only count + split are charged
    per_level_scale = granulation_target / (max_depth * unit_per_level)

    eps_cnt = 0.1 * per_level_scale
    eps_spl = 0.1 * per_level_scale
    eps_rad = 0.0      # public
    eps_ctr = 0.0      # public
    eps_count_release = max(float(count_release), 1e-6)

    rng = np.random.default_rng(seed)

    scores = []
    include_flags = []
    min_dists = []

    for trial in range(n_trials):
        try:
            include, X_in, y_in = _make_corner_canary_trial(
                X_arr=X_arr,
                y_arr=y_arr,
                rng=rng,
                trial_size=trial_size,
                canary_x=canary_x,
                canary_y=canary_y,
                class_indices=class_indices,
                classes=classes
            )
        except Exception:
            continue

        X_in_df = pd.DataFrame(
            X_in,
            columns=[f"f{i}" for i in range(d)]
        )
        y_in_series = pd.Series(y_in.astype(int))

        try:
            X_priv, _, _ = run_dp_synthetic_release_no_split_scaled(
                X_scaled_df=X_in_df,
                y_series=y_in_series,
                seed=trial,
                max_depth=max_depth,
                eps_count_per_level=eps_cnt,
                eps_split_per_level=eps_spl,
                eps_radius_per_level=eps_rad,
                eps_center_per_level=eps_ctr,
                eps_count_release=eps_count_release
                # delta is left at its default (0.0)
            )
        except Exception:
            continue

        if len(X_priv) == 0:
            continue

        released = np.asarray(X_priv.values, dtype=float)

        dists = np.linalg.norm(released - canary_x.reshape(1, -1), axis=1)
        min_dist = float(np.min(dists))

        scores.append(-min_dist)
        include_flags.append(include)
        min_dists.append(min_dist)

    return (
        np.asarray(scores, dtype=float),
        np.asarray(include_flags, dtype=bool),
        np.asarray(min_dists, dtype=float)
    )


# ----------------------------------------------------------------------------
# Main audit function
# ----------------------------------------------------------------------------
def empirical_eps_audit(
    X_scaled,
    y,
    eps_config,
    n_trials,
    seed,
    trial_size=200,
    n_thresholds=20
):
    scores, include_flags, min_dists = _run_canary_audit_core(
        X_scaled=X_scaled,
        y=y,
        eps_config=eps_config,
        n_trials=n_trials,
        seed=seed,
        trial_size=trial_size
    )

    n_valid = len(scores)
    n_with_total = int(np.sum(include_flags)) if n_valid > 0 else 0
    n_without_total = int(np.sum(~include_flags)) if n_valid > 0 else 0

    result = {
        "eps_lower_bound": 0.0,
        "threshold": None,
        "tp": 0,
        "fp": 0,
        "n_with": n_with_total,
        "n_without": n_without_total,
        "n_valid_trials": n_valid
    }

    if n_valid == 0 or n_with_total == 0 or n_without_total == 0:
        return result

    quantiles = np.quantile(scores, np.linspace(0.05, 0.95, n_thresholds))
    thresholds = np.unique(quantiles[np.isfinite(quantiles)])

    best_eps = 0.0
    best_thresh = None
    best_tp = 0
    best_fp = 0

    for thresh in thresholds:
        guess_in = scores >= thresh

        tp = int(np.sum(guess_in & include_flags))
        fp = int(np.sum(guess_in & ~include_flags))

        tpr_lo, _ = clopper_pearson(tp, n_with_total)
        _, fpr_hi = clopper_pearson(fp, n_without_total)

        tpr_lo = max(float(tpr_lo), 1e-4)
        fpr_hi = max(float(fpr_hi), 1e-4)

        eps_lower = float(np.log(tpr_lo / fpr_hi))

        if eps_lower > best_eps:
            best_eps = eps_lower
            best_thresh = float(thresh)
            best_tp = tp
            best_fp = fp

    result["eps_lower_bound"] = max(float(best_eps), 0.0)
    result["threshold"] = best_thresh
    result["tp"] = best_tp
    result["fp"] = best_fp

    return result


# ----------------------------------------------------------------------------
# Debug version
# ----------------------------------------------------------------------------
def empirical_eps_audit_debug(
    X_scaled,
    y,
    dataset_name,
    eps_config,
    n_trials,
    seed,
    trial_size=200
):
    scores, include_flags, min_dists = _run_canary_audit_core(
        X_scaled=X_scaled,
        y=y,
        eps_config=eps_config,
        n_trials=n_trials,
        seed=seed,
        trial_size=trial_size
    )

    if len(scores) == 0:
        print(f"  {dataset_name}: no valid audit trials")
        return

    scores = np.asarray(scores)
    include_flags = np.asarray(include_flags, dtype=bool)
    min_dists = np.asarray(min_dists)

    if np.sum(include_flags) == 0 or np.sum(~include_flags) == 0:
        print(f"  {dataset_name}: only one trial condition survived")
        return

    print(
        f"  {dataset_name}: included  "
        f"mean_score={scores[include_flags].mean():.4f}  "
        f"mean_min_dist={min_dists[include_flags].mean():.4f}  "
        f"std_min_dist={min_dists[include_flags].std():.4f}"
    )
    print(
        f"  {dataset_name}: excluded  "
        f"mean_score={scores[~include_flags].mean():.4f}  "
        f"mean_min_dist={min_dists[~include_flags].mean():.4f}  "
        f"std_min_dist={min_dists[~include_flags].std():.4f}"
    )


# ----------------------------------------------------------------------------
# Run audit on all datasets — now with public bounds, not MinMaxScaler
# ----------------------------------------------------------------------------
AUDIT_EPS = float(globals().get("TOTAL_EPS", 3.0))

# Public bounds from schema; no privacy budget spent
scaled_datasets = {}
for ds_name, (X, y) in datasets.items():
    lo, hi = public_bounds_for(ds_name, X)
    X_scaled = scale_with_public_bounds(X, lo, hi)
    scaled_datasets[ds_name] = (X_scaled, y)

# Fix 5: delta = 0.0 for pure (eps,0)-DP
max_depth = 3
delta = 0.0                # <-- changed from 1e-5
claimed_delta = max_depth * delta   # will be 0.0

audit_results = []

for ds_name, (X_scaled, y) in scaled_datasets.items():
    res = empirical_eps_audit(
        X_scaled=X_scaled,
        y=y,
        eps_config=AUDIT_EPS,
        n_trials=500,
        seed=SEED,
        trial_size=200,
        n_thresholds=20
    )
    res["Dataset"] = ds_name
    res["claimed_eps"] = AUDIT_EPS
    res["claimed_delta"] = claimed_delta
    audit_results.append(res)

df_audit = pd.DataFrame(audit_results)

cols = [
    "Dataset",
    "claimed_eps",
    "claimed_delta",
    "eps_lower_bound",
    "threshold",
    "tp",
    "fp",
    "n_with",
    "n_without",
    "n_valid_trials"
]

df_display = df_audit[cols].copy()
df_display["claimed_delta"] = df_display["claimed_delta"].apply(lambda x: f"{x:.1e}")
for c in ["claimed_eps", "eps_lower_bound", "threshold"]:
    df_display[c] = df_display[c].astype(float).round(3)
print(df_display.to_string(index=False))

print(
    "\n⚠️ DISCLAIMER: The 'eps_lower_bound' values above are NOT valid lower bounds "
    "on differential privacy. This audit is underpowered and is presented only "
    "to document the attempted canary test. See positive control below."
)

# ----------------------------------------------------------------------------
# Positive control: run the same audit with very high epsilon (near-noiseless)
# ----------------------------------------------------------------------------
print("\n--- Positive control (eps_config = 1e6) ---")
powerful = False
for ds_name, (X_scaled, y) in scaled_datasets.items():
    pc_res = empirical_eps_audit(
        X_scaled=X_scaled,
        y=y,
        eps_config=1e6,
        n_trials=500,
        seed=SEED,
        trial_size=200,
        n_thresholds=20
    )
    print(
        f"{ds_name}: eps_lower_bound={pc_res['eps_lower_bound']:.3f} "
        f"(tp={pc_res['tp']}, fp={pc_res['fp']})"
    )
    if pc_res["eps_lower_bound"] > 0.1:
        powerful = True
        print(f"  --> This dataset shows detectable signal; audit may have some power.")
    else:
        print(f"  --> No power: canary/statistic cannot detect membership even with no noise.")

if not powerful:
    print("\n🔴 NONE of the datasets show detection power in the positive control.")
    print("   This audit is completely uninformative for privacy analysis.")
    print("   The results above must NOT be used to claim any privacy guarantee.")
else:
    print("\nPositive control has power on at least one dataset; interpret main results with caution.")

# ----------------------------------------------------------------------------
# Debug runs on all datasets (optional, can be commented out to reduce output)
# ----------------------------------------------------------------------------
for ds_name, (X_scaled, y) in scaled_datasets.items():
    print(ds_name)
    empirical_eps_audit_debug(
        X_scaled=X_scaled,
        y=y,
        dataset_name=ds_name,
        eps_config=AUDIT_EPS,
        n_trials=500,
        seed=SEED,
        trial_size=200
    )

In [ ]:
# =============================================================================
# Isolation test: sklearn vs dp_sgd_logreg with sigma floor (0.1)
# This proves gradient clipping does NOT cause significant utility loss.
# =============================================================================
for ds_name, (X, y) in datasets.items():
    if y.nunique() != 2:
        continue
    try:
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    except ValueError:
        continue
    scaler = MinMaxScaler()
    X_tr_n = scaler.fit_transform(X_tr)
    X_te_n = scaler.transform(X_te)
    y_min = y.min()

    # 1. Sklearn baseline
    sk = LogisticRegression(max_iter=2000).fit(X_tr_n, y_tr)
    acc_sk = balanced_accuracy_score(y_te, sk.predict(X_te_n))

    # 2. DP-SGD with minimal noise (eps huge, sigma clamped at 0.1)
    rng = np.random.default_rng(42)
    w, sigma = dp_sgd_logreg(X_tr_n, y_tr.values, eps_target=1e15, delta=1e-5,
                             T=100, clip_norm=1.0, lr=0.5, rng=rng)
    # predict
    preds = dp_sgd_predict_proba(w, X_te_n)
    if preds.ndim == 2:
        preds = preds.argmax(axis=1) + y_min
    else:
        preds = (preds > 0.5).astype(int) + y_min
    acc_dp = balanced_accuracy_score(y_te, preds)

    print(f"{ds_name}: Sklearn={acc_sk:.3f}  DP-SGD(sigma={sigma:.4f})={acc_dp:.3f}  "
          f"Δ={acc_sk-acc_dp:.4f}")
print("\nConclusion: Clipping cost is minimal; the sigma floor (0.1) still yields high utility.")

In [ ]:
# Cell 28a (NEW): KD-tree-style DP baseline — reimplementation of the GENERAL
# RECIPE in Kreačić, Nouri, Potluru, Balch, Veloso, "Differentially Private
# Synthetic Data Using KD-Trees," UAI 2023, PMLR 216:1143-1153. This is NOT
# their verified official code — it is our best-effort reimplementation of
# the space-partitioning + noisy-count + box-sampling recipe described in
# their abstract, built by reusing our own already-verified DP primitives,
# for an internal, clearly-labeled comparison only. Do not report these
# numbers as a reproduction of their published results.
#
# Structural difference from DP-GBC (dp_generate_granular_balls_with_radius):
#   - DP-GBC splits are chosen by a LABEL-PURITY score (majority-count).
#   - This KD-tree baseline splits are chosen by a DENSITY-BALANCE score
#     (how evenly a threshold divides the node's point count), which does
#     not look at y at all -- consistent with KD-trees' general-purpose,
#     not classification-specific, design goal.
#   - Stopping is by count/depth only (no purity early-stop), since this
#     variant partitions the FEATURE SPACE, not label homogeneity.
# Same sensitivity-1 exponential-mechanism machinery as DP-GBC is reused,
# so the privacy accounting is identical in form.

def balance_score(vals, t):
    """Sensitivity 1: adding/removing one record shifts n_left or n_right
    by exactly 1, so |n_left - n_right| changes by at most 1."""
    left = vals <= t
    n_left, n_right = int(left.sum()), int((~left).sum())
    if n_left < 2 or n_right < 2:
        return -1e9
    return -abs(n_left - n_right)

def dp_generate_kdtree_balls(X, y, max_depth, eps_count_per_level,
                              eps_split_per_level, min_size=10,
                              n_candidate_thresholds=8, rng=None,
                              all_classes=None):
    rng = rng or np.random.default_rng(SEED)
    X_arr = X.values if hasattr(X, "values") else np.asarray(X)
    y_arr = np.asarray(y)
    global_classes = all_classes if all_classes is not None else np.unique(y_arr)
    n_features = X_arr.shape[1]
    balls = []
    root_lo, root_hi = np.zeros(n_features), np.ones(n_features)
    queue = [(np.arange(len(X_arr)), 0, root_lo, root_hi)]

    while queue:
        indices, depth, box_lo, box_hi = queue.pop(0)
        X_node, y_node = X_arr[indices], y_arr[indices]

        counts_full = np.array([np.sum(y_node == c) for c in global_classes])
        noisy_counts = np.array([
            laplace_mech(c, sensitivity=1.0, eps=eps_count_per_level, rng=rng)
            for c in counts_full
        ])
        noisy_counts = np.clip(noisy_counts, 0, None) + 1.0

        stop = (noisy_counts.sum() < min_size) or (depth >= max_depth)  # no purity check

        def make_leaf():
            center = (box_lo + box_hi) / 2.0
            radius = np.linalg.norm(box_hi - box_lo) / 2.0
            balls.append({
                "indices": indices, "center": center, "radius": radius,
                "box_lo": box_lo.copy(), "box_hi": box_hi.copy(),
                "n": len(indices), "classes": global_classes,
                "noisy_class_counts": np.clip(noisy_counts, 0, None)
            })

        if stop:
            make_leaf(); continue

        candidates, scores = [], []
        for f in range(n_features):
            lo_f, hi_f = box_lo[f], box_hi[f]
            if hi_f - lo_f < 1e-9:
                continue
            vals = X_node[:, f]
            thresholds = np.linspace(lo_f, hi_f, n_candidate_thresholds + 2)[1:-1]
            for t in thresholds:
                candidates.append((f, t))
                scores.append(balance_score(vals, t))

        if not candidates:
            make_leaf(); continue

        chosen_f, chosen_t = exponential_mechanism(candidates, scores, 1.0, eps_split_per_level, rng)
        left_mask = X_node[:, chosen_f] <= chosen_t

        left_lo, left_hi = box_lo.copy(), box_hi.copy()
        left_hi[chosen_f] = np.clip(min(left_hi[chosen_f], chosen_t), box_lo[chosen_f], box_hi[chosen_f])
        right_lo, right_hi = box_lo.copy(), box_hi.copy()
        right_lo[chosen_f] = np.clip(max(right_lo[chosen_f], chosen_t), box_lo[chosen_f], box_hi[chosen_f])

        queue.append((indices[left_mask], depth + 1, left_lo, left_hi))
        queue.append((indices[~left_mask], depth + 1, right_lo, right_hi))

    return balls


def run_dp_kdtree_pipeline(X, y, seed, lo, hi, max_depth=3,
                            eps_count_per_level=0.1, eps_split_per_level=0.1,
                            eps_count_release=2.4, all_classes=None,
                            precomputed_split=None, sampling_mode="uniform"):
    """Same budget-accounting shape as run_dp_gbc_pipeline, for a fair,
    matched-epsilon comparison. eps_total = D*(eps_count+eps_split) + eps_count_release."""
    rng = np.random.default_rng(seed)
    ledger = PrivacyLedger()

    if precomputed_split is not None:
        X_tr, X_te, y_tr, y_te = precomputed_split
    else:
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)

    X_tr_n = scale_with_public_bounds(X_tr, lo, hi)
    X_te_n = scale_with_public_bounds(X_te, lo, hi)

    balls = dp_generate_kdtree_balls(
        X_tr_n, y_tr.values, max_depth=max_depth,
        eps_count_per_level=eps_count_per_level,
        eps_split_per_level=eps_split_per_level,
        rng=rng, all_classes=all_classes
    )

    per_level = eps_count_per_level + eps_split_per_level
    for depth in range(max_depth):
        ledger.spend_sequential_charged_once(per_level, delta=0.0, n_branches=len(balls),
                                              note=f"kdtree granulation depth {depth+1}")
    ledger.spend_sequential(eps_count_release, 0.0, note="kdtree per-leaf synthetic-size release")

    sizes = dp_release_ball_counts(balls, eps_count_release, rng)
    X_synth, y_synth = dp_sample_synthetic_records(
        balls, sizes, rng, d=X_tr_n.shape[1], sampling_mode=sampling_mode
    )
    return pd.DataFrame(X_synth, columns=X_tr_n.columns), X_te_n, pd.Series(y_synth), y_te, ledger, balls

In [ ]:
# Cell 28b (NEW): Run the KD-tree-style baseline on the same 14 datasets,
# same seeds, same TOTAL_EPS as the main DP-GBC results (Cell 7), for a
# matched-budget comparison table.

kdtree_results = []
_max_depth = 3
_eps_count = _eps_split = 0.1
_eps_count_release = TOTAL_EPS - _max_depth * (_eps_count + _eps_split)

for ds_name in NUMERIC_DATASETS:
    if ds_name not in datasets:
        continue
    X, y = datasets[ds_name]
    lo, hi = public_bounds_for(ds_name, X)

    for seed in SEEDS:
        try:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X, y, test_size=0.3, random_state=seed, stratify=y
            )
        except ValueError:
            continue
        fixed_split = (X_tr, X_te, y_tr, y_te)
        X_tr_n = scale_with_public_bounds(X_tr, lo, hi)

        X_tr_kd, X_te_kd, y_tr_kd, y_te_kd, ledger_kd, balls_kd = run_dp_kdtree_pipeline(
            X, y, seed, lo, hi, max_depth=_max_depth,
            eps_count_per_level=_eps_count, eps_split_per_level=_eps_split,
            eps_count_release=_eps_count_release,
            all_classes=classes_public[ds_name],
            precomputed_split=fixed_split,
        )
        assert abs(ledger_kd.total_eps - TOTAL_EPS) < 1e-6

        if len(X_tr_kd) == 0:
            continue

        util_kd, model_kd = evaluate_utility(X_tr_kd, y_tr_kd, X_te_kd, y_te_kd)
        mia_kd, _ = membership_inference_attack_rf_synth(
            X_tr_kd, y_tr_kd, X_tr_n, y_tr, X_te_kd, y_te, seed
        )

        kdtree_results.append({
            "Dataset": ds_name, "Seed": seed,
            "kdtree_bal_acc": util_kd["balanced_accuracy"],
            "kdtree_mia_rf": mia_kd,
        })
    print(f"done kdtree-baseline: {ds_name}")

df_kdtree = pd.DataFrame(kdtree_results)
df_kdtree.to_csv("dpgbc_kdtree_baseline.csv", index=False)

merged = df_kdtree.merge(
    df_main[["Dataset", "Seed", "flat_LR_bal_acc", "dplc_bal_acc", "dplc_MIA_rf"]],
    on=["Dataset", "Seed"]
)
comp_kdtree = merged.groupby("Dataset")[
    ["flat_LR_bal_acc", "kdtree_bal_acc", "dplc_bal_acc", "kdtree_mia_rf", "dplc_MIA_rf"]
].mean().round(3)

print("\n=== DP-GBC vs. KD-tree-style baseline (matched epsilon=3.0) ===")
print(comp_kdtree.to_string())

from scipy.stats import wilcoxon as _wilcoxon2
piv2 = merged.groupby("Dataset")[["kdtree_bal_acc", "dplc_bal_acc"]].mean().dropna()
stat, p = _wilcoxon2(piv2["dplc_bal_acc"], piv2["kdtree_bal_acc"])
print(f"\nWilcoxon, DP-GBC(+LR) vs. KD-tree-baseline utility, p={p:.5f}, "
      f"median diff = {np.median(piv2['dplc_bal_acc'] - piv2['kdtree_bal_acc']):.4f}")

In [ ]:
# ==============================================================
# Publication-quality figures for the DP-GBC paper (IEEE conference)
# Requires: df_main (Cell 7), df_sweep (Cell 8), tradeoff (Cell 11/12),
#           df_fidelity (Fidelity cell), NUMERIC_DATASETS, valid_datasets,
#           df_sampling_ablation (Sampling ablation cell),
#           comp (KD-tree comparison cell)
# Outputs: figures/fig{1..8}.pdf (vector, for LaTeX) + .png (preview)
# ==============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import seaborn as sns
from scipy.stats import t as t_dist

os.makedirs("figures", exist_ok=True)

# -----------------------------------------------------------------
# Dependency guard: fail early if required variables are not defined
# -----------------------------------------------------------------
_REQUIRED_VARS = ["df_main", "df_sweep", "tradeoff", "df_fidelity",
                  "NUMERIC_DATASETS", "valid_datasets", "df_sampling_ablation",
                  "comp"]
_missing = [v for v in _REQUIRED_VARS if v not in globals()]
if _missing:
    raise NameError(
        f"Figures script requires these already defined in this session: "
        f"{_missing}. Run Cells 7, 8, 9, 11/12, the Fidelity cell, "
        f"the Sampling ablation cell, and the KD-tree comparison cell first."
    )

# ---------------- IEEE-compliant global style ----------------
# IEEE Author Center: 1-col = 3.5in, 2-col = 7.16in, >=300 DPI,
# embedded fonts, colorblind-safe (verified against IEEE's own
# "Improve Your Graphics" / "Resolution and Size" pages).
COL1, COL2 = 3.5, 7.16
DPI = 300

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 9,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 6.5,
    "legend.frameon": False,
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,          # embed as real fonts, not paths
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.5,
    "axes.axisbelow": True,
})

# Colorblind-safe palette + a linestyle/marker per method so the
# figure still reads correctly in greyscale (IEEE explicitly asks
# for this).
PAL = sns.color_palette("colorblind")
METHOD_STYLE = {
    "Flat-LR":         dict(color=PAL[7], hatch="//",  marker="o"),
    "Flat-RF":         dict(color=PAL[1], hatch="\\\\", marker="s"),
    "DP-GBC (direct)": dict(color=PAL[0], hatch="",    marker="^"),
    "DP-GBC (+LR)":    dict(color=PAL[2], hatch="xx",  marker="D"),
}

def save_fig(fig, name):
    fig.savefig(f"figures/{name}.pdf")
    fig.savefig(f"figures/{name}.png")
    plt.close(fig)
    print(f"saved figures/{name}.pdf and .png")


# ==============================================================
# FIGURE 1 — DP-GBC mechanism workflow (schematic, single column)
# ==============================================================
def fig1_mechanism_workflow():
    fig, ax = plt.subplots(figsize=(COL2, 1.9))
    ax.set_xlim(0, 10); ax.set_ylim(0, 2.2); ax.axis("off")

    steps = [
        ("Private\ngranulation",
         r"Exp. mech. selects split" "\n" r"Laplace noise on counts",
         PAL[0]),
        ("Noisy count\nrelease",
         r"$\tilde n_i = n_i + \mathrm{Lap}(1/\varepsilon_{\mathrm{rel}})$",
         PAL[2]),
        ("Synthetic\nsampling",
         r"Uniform in box $[\ell_i,h_i]$" "\n" r"labels $\sim \tilde n_i$",
         PAL[3]),
    ]
    box_w, box_h, gap = 2.6, 1.5, 0.55
    x0 = 0.3
    for i, (title, detail, color) in enumerate(steps):
        x = x0 + i * (box_w + gap)
        box = FancyBboxPatch((x, 0.35), box_w, box_h,
                              boxstyle="round,pad=0.08,rounding_size=0.12",
                              linewidth=1.1, edgecolor="black",
                              facecolor=color, alpha=0.18)
        ax.add_patch(box)
        ax.text(x + box_w/2, 0.35 + box_h - 0.28, title,
                 ha="center", va="top", fontsize=8, fontweight="bold")
        ax.text(x + box_w/2, 0.35 + box_h/2 - 0.15, detail,
                 ha="center", va="center", fontsize=6.8)
        if i < len(steps) - 1:
            ax.add_patch(FancyArrowPatch(
                (x + box_w + 0.06, 0.35 + box_h/2),
                (x + box_w + gap - 0.06, 0.35 + box_h/2),
                arrowstyle="-|>", mutation_scale=12, linewidth=1.1))
    ax.text(x0 + 1.5*(box_w+gap), 2.05,
             r"total budget $= D(\varepsilon_{\mathrm{cnt}}+\varepsilon_{\mathrm{spl}}) + \varepsilon_{\mathrm{rel}}$",
             ha="center", fontsize=6.5, style="italic")
    save_fig(fig, "fig1_mechanism_workflow")


# ==============================================================
# FIGURE 2 — Utility comparison across datasets (bar + 95% CI)
# ==============================================================
def fig2_utility_comparison(df_main, datasets_order=None):
    methods = {
        "flat_LR_bal_acc": "Flat-LR",
        "flat_RF_bal_acc": "Flat-RF",
        "dpgbc_bal_acc":   "DP-GBC (direct)",
        "dplc_bal_acc":    "DP-GBC (+LR)",
    }
    datasets_order = datasets_order or sorted(df_main["Dataset"].unique())

    def mean_ci(x, conf=0.95):
        x = np.asarray(x, dtype=float); n = len(x)
        m = x.mean()
        if n < 2:
            return m, 0.0
        se = x.std(ddof=1) / np.sqrt(n)
        h = se * t_dist.ppf((1 + conf) / 2., n - 1)
        return m, h

    means = {m: [] for m in methods}
    errs = {m: [] for m in methods}
    for ds in datasets_order:
        sub = df_main[df_main["Dataset"] == ds]
        for col in methods:
            m, h = mean_ci(sub[col].values)
            means[col].append(m)
            errs[col].append(h)

    n_ds, n_m = len(datasets_order), len(methods)
    x = np.arange(n_ds)
    width = 0.8 / n_m

    fig, ax = plt.subplots(figsize=(COL2, 2.6))
    for i, (col, label) in enumerate(methods.items()):
        st = METHOD_STYLE[label]
        ax.bar(x + (i - (n_m-1)/2) * width, means[col], width,
               yerr=errs[col], capsize=1.5, label=label,
               color=st["color"], hatch=st["hatch"],
               edgecolor="black", linewidth=0.4,
               error_kw=dict(elinewidth=0.6, capthick=0.6))

    ax.set_xticks(x)
    ax.set_xticklabels(datasets_order, rotation=45, ha="right")
    ax.set_ylabel("Balanced accuracy")
    ax.set_ylim(0, 1.05)
    ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.18))
    ax.set_title("")
    save_fig(fig, "fig2_utility_comparison")


# ==============================================================
# FIGURE 3 — Privacy-utility trade-off (quadrant scatter)
# ==============================================================
def fig3_tradeoff_scatter(tradeoff):
    fig, ax = plt.subplots(figsize=(COL1, 3.0))

    sc = ax.scatter(tradeoff["util_cost"], tradeoff["mia_gain"],
                     c=tradeoff["flat_mia"], cmap="viridis",
                     s=55, edgecolor="black", linewidth=0.5, zorder=3)

    ax.axhline(0, color="grey", lw=0.8, ls="--", zorder=1)
    ax.axvline(0, color="grey", lw=0.8, ls="--", zorder=1)

    # quadrant labels
    ax.text(ax.get_xlim()[1]*0.55, ax.get_ylim()[1]*0.92,
             "utility loss,\nprivacy gain", fontsize=6, ha="center",
             style="italic", color="dimgrey")
    ax.text(ax.get_xlim()[0]*0.6 if ax.get_xlim()[0] < 0 else 0.02,
             ax.get_ylim()[1]*0.92,
             "utility gain,\nprivacy gain", fontsize=6, ha="center",
             style="italic", color="dimgrey")

    # label the extreme / most interesting points
    for _, row in tradeoff.nlargest(3, "mia_gain").iterrows():
        ax.annotate(row["Dataset"], (row["util_cost"], row["mia_gain"]),
                    textcoords="offset points", xytext=(4, 4), fontsize=6)

    cbar = fig.colorbar(sc, ax=ax, pad=0.02, fraction=0.05)
    cbar.set_label("Flat-noise MIA AUC\n(attack strength)", fontsize=6.5)
    cbar.ax.tick_params(labelsize=6)

    ax.set_xlabel(r"Utility cost  $\Delta$balanced accuracy (DP-GBC $-$ flat)")
    ax.set_ylabel(r"MIA advantage gain  (|AUC$-$0.5| flat $-$ DP-GBC)")
    save_fig(fig, "fig3_tradeoff_scatter")


# ==============================================================
# FIGURE 4 — Epsilon sweep trend (dual-axis line, with win-frac)
# ==============================================================
def fig4_epsilon_sweep(df_sweep):
    summary = df_sweep.groupby("TOTAL_EPS").agg(
        mean_util_gap=("util_gap", "mean"),
        mean_mia_gap=("mia_gap", "mean"),
    ).reset_index()
    summary["win_frac"] = df_sweep.groupby("TOTAL_EPS").apply(
        lambda g: ((g["util_gap"] > 0) & (g["mia_gap"] < 0)).mean()
    ).values

    fig, ax1 = plt.subplots(figsize=(COL1, 2.6))
    ax2 = ax1.twinx()

    l1, = ax1.plot(summary["TOTAL_EPS"], summary["mean_util_gap"],
                    marker="^", color=PAL[0], label="Utility gap (DP-GBC$-$flat)")
    l2, = ax2.plot(summary["TOTAL_EPS"], summary["mean_mia_gap"],
                    marker="s", ls="--", color=PAL[3], label="MIA advantage gap")

    # (removed: an invisible alpha=0 bar and an unused twiny() axis that never
    #  drew anything -- win_frac is already shown via annotate() below, and
    #  x-ticks are set explicitly a few lines down.)

    ax1.axhline(0, color="grey", lw=0.6, ls=":")
    ax2.axhline(0, color="grey", lw=0.6, ls=":")

    ax1.set_xlabel(r"Privacy budget $\varepsilon$")
    ax1.set_ylabel("Mean utility gap", color=PAL[0])
    ax2.set_ylabel("Mean MIA advantage gap", color=PAL[3])
    ax1.tick_params(axis="y", labelcolor=PAL[0])
    ax2.tick_params(axis="y", labelcolor=PAL[3])
    ax1.set_xscale("log")
    ax1.set_xticks(summary["TOTAL_EPS"])
    ax1.set_xticklabels([str(v) for v in summary["TOTAL_EPS"]])

    for x, w in zip(summary["TOTAL_EPS"], summary["win_frac"]):
        ax1.annotate(f"{w:.0%}", (x, ax1.get_ylim()[1]*0.92),
                     ha="center", fontsize=6, color="dimgrey")

    lines = [l1, l2]
    ax1.legend(lines, [l.get_label() for l in lines],
               loc="lower left", fontsize=6)
    save_fig(fig, "fig4_epsilon_sweep")


# ==============================================================
# FIGURE 5 — Fidelity heatmap
# ==============================================================
def fig5_fidelity_heatmap(df_fidelity):
    summary = df_fidelity.groupby("Dataset")[
        ["marginal_ks", "marginal_wasserstein", "correlation_mae", "tstr_minus_trtr"]
    ].mean()

    # min-max normalize each column to [0,1] so colors are comparable
    # across metrics with very different natural scales; absolute
    # values still go in the annotation text.
    norm = (summary - summary.min()) / (summary.max() - summary.min() + 1e-12)
    # for tstr_minus_trtr, more negative = worse, so flip sign before
    # normalizing so "red" always means "worse" consistently
    norm["tstr_minus_trtr"] = 1 - norm["tstr_minus_trtr"]

    fig, ax = plt.subplots(figsize=(COL1, 3.4))
    sns.heatmap(norm, annot=summary.round(2), fmt="", cmap="RdYlGn_r",
                cbar_kws={"label": "relative (worse \u2192 better) ", "shrink": 0.7},
                linewidths=0.4, linecolor="white",
                annot_kws={"fontsize": 5.5}, ax=ax)
    ax.set_xticklabels(["KS\n(marginals)", "Wasserstein\n(marginals)",
                         "Corr. MAE", "TSTR\u2212TRTR"], fontsize=6.5)
    ax.set_ylabel("")
    save_fig(fig, "fig5_fidelity_heatmap")


# ==============================================================
# FIGURE 6 — MIA advantage + Pareto dominance (2-panel, combined)
# ==============================================================
def fig6_mia_and_dominance(df_main, valid_datasets):
    df_valid = df_main[df_main["Dataset"].isin(valid_datasets)]

    mia = df_valid.groupby("Dataset")[["flat_LR_MIA_rf", "dplc_MIA_rf"]].mean()
    mia = mia.sort_values("flat_LR_MIA_rf", ascending=False)

    dominance = df_valid.groupby("Dataset").apply(
        lambda g: ((g["dplc_bal_acc"] >= g["flat_LR_bal_acc"]) &
                   ((g["dplc_MIA_rf"] - 0.5).abs() <=
                    (g["flat_LR_MIA_rf"] - 0.5).abs())).mean()
    ).sort_values(ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.7))

    # --- panel (a): MIA AUC, flat vs DP-GBC ---
    ax = axes[0]
    x = np.arange(len(mia))
    ax.bar(x - 0.19, mia["flat_LR_MIA_rf"], 0.38, label="Flat-LR",
           color=PAL[7], hatch="//", edgecolor="black", linewidth=0.4)
    ax.bar(x + 0.19, mia["dplc_MIA_rf"], 0.38, label="DP-GBC (+LR)",
           color=PAL[2], hatch="xx", edgecolor="black", linewidth=0.4)
    ax.axhline(0.5, color="grey", lw=0.8, ls="--")
    ax.set_xticks(x); ax.set_xticklabels(mia.index, rotation=60, ha="right", fontsize=6)
    ax.set_ylabel("MIA AUC (0.5 = chance)")
    ax.set_ylim(0.3, 1.05)
    ax.legend(fontsize=6)
    ax.set_title("(a) Membership-inference advantage", fontsize=7.5)

    # --- panel (b): Pareto dominance fraction ---
    ax = axes[1]
    y = np.arange(len(dominance))
    ax.barh(y, dominance.values, color=PAL[0], edgecolor="black", linewidth=0.4)
    ax.set_yticks(y); ax.set_yticklabels(dominance.index, fontsize=6)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.05)
    ax.set_xlabel("Fraction of seeds: DP-GBC dominates\n(utility \u2265 flat AND MIA \u2264 flat)")
    ax.set_title("(b) Pareto dominance per dataset", fontsize=7.5)

    fig.tight_layout()
    save_fig(fig, "fig6_mia_and_dominance")


# ==============================================================
# FIGURE 7 — Sampling-geometry ablation (uniform vs. gaussian)
# ==============================================================
def fig7_sampling_ablation(df_sampling_ablation):
    piv = df_sampling_ablation.groupby(["Dataset", "sampling_mode"])[
        ["marginal_ks", "correlation_mae"]
    ].mean().unstack("sampling_mode")
    order = piv[("marginal_ks", "uniform")].sort_values(ascending=False).index

    fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.6))
    for ax, metric, title in zip(
        axes, ["marginal_ks", "correlation_mae"],
        ["(a) Marginal KS error", "(b) Correlation MAE"]
    ):
        x = np.arange(len(order))
        ax.bar(x - 0.19, piv.loc[order, (metric, "uniform")], 0.38,
               label="Box-uniform", color=PAL[0], hatch="//",
               edgecolor="black", linewidth=0.4)
        ax.bar(x + 0.19, piv.loc[order, (metric, "gaussian")], 0.38,
               label="Box-Gaussian", color=PAL[3], hatch="xx",
               edgecolor="black", linewidth=0.4)
        ax.set_xticks(x); ax.set_xticklabels(order, rotation=60, ha="right", fontsize=6)
        ax.set_title(title, fontsize=7.5)
        ax.legend(fontsize=6)
    fig.tight_layout()
    save_fig(fig, "fig7_sampling_ablation")


# ==============================================================
# FIGURE 8 — DP-GBC vs. KD-tree-style baseline (matched epsilon)
# ==============================================================
def fig8_kdtree_comparison(comp):
    fig, ax = plt.subplots(figsize=(COL2, 2.6))
    x = np.arange(len(comp))
    ax.bar(x - 0.25, comp["flat_LR_bal_acc"], 0.25, label="Flat-LR",
           color=PAL[7], hatch="//", edgecolor="black", linewidth=0.4)
    ax.bar(x, comp["kdtree_bal_acc"], 0.25, label="KD-tree-style baseline",
           color=PAL[4], hatch="..", edgecolor="black", linewidth=0.4)
    ax.bar(x + 0.25, comp["dplc_bal_acc"], 0.25, label="DP-GBC (+LR)",
           color=PAL[2], hatch="xx", edgecolor="black", linewidth=0.4)
    ax.set_xticks(x); ax.set_xticklabels(comp.index, rotation=45, ha="right")
    ax.set_ylabel("Balanced accuracy")
    ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.18), fontsize=6.5)
    save_fig(fig, "fig8_kdtree_comparison")


# ==============================================================
# Run all
# ==============================================================
if __name__ == "__main__":
    fig1_mechanism_workflow()
    fig2_utility_comparison(df_main, datasets_order=NUMERIC_DATASETS)
    fig3_tradeoff_scatter(tradeoff)
    fig4_epsilon_sweep(df_sweep)
    fig5_fidelity_heatmap(df_fidelity)
    fig6_mia_and_dominance(df_main, valid_datasets)
    fig7_sampling_ablation(df_sampling_ablation)
    fig8_kdtree_comparison(comp)
    print("\nAll figures written to ./figures/  (use the .pdf files in LaTeX)")